In [1]:
!pip install playwright python-dotenv pandas
!python -m playwright install

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/37.9 MB ? eta -:--:--
    --------------------------------------- 0.8/37.9 MB 2.4 MB/s eta 0:00:16
   -- ------------------------------------- 2.1/37.9 MB 3.8 MB/s eta 0:00:10
   ---- ----------------------------------- 4.2/37.9 MB 6.0 MB/s eta 0:00:06
   ------ --------------------------------- 6.0/37.9 MB 7.4 MB/s eta 0:00:05
   --------- ------------------------------ 9.2/37.9 MB 8.0 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/37.9 MB 8.5 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/37.9 MB 8.5 MB/s eta 0:00:04
   ---------- ----------------------------- 10.0/37.9 MB 8.5 MB/s eta 0:00:04
   ------------ -------

In [2]:
env_content = """
PORTAL_USERNAME=2380223
PORTAL_PASSWORD=Muneeb@Anjum3543
"""

with open(".env", "w", encoding="utf-8") as file:
    file.write(env_content.strip())

print(".env file created successfully.")

.env file created successfully.


In [3]:
gitignore_content = """
.env
__pycache__/
*.pkl
*.html
*.png
"""

with open(".gitignore", "w", encoding="utf-8") as file:
    file.write(gitignore_content.strip())

print(".gitignore file created successfully.")

.gitignore file created successfully.


In [4]:
scraper_code = r'''
import time
from playwright.sync_api import sync_playwright

PORTAL_URL = "https://springzabdesk.szabist-isb.edu.pk/"

def main():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page()

        page.goto(PORTAL_URL)
        print("Portal opened.")

        print("Login manually in the browser.")
        print("After login, go to Attendance / Marks page.")
        print("You have 120 seconds.")

        time.sleep(120)

        page.screenshot(path="portal_after_login.png", full_page=True)

        html = page.content()
        with open("portal_after_login.html", "w", encoding="utf-8") as file:
            file.write(html)

        print("Saved portal_after_login.png")
        print("Saved portal_after_login.html")

        browser.close()

if __name__ == "__main__":
    main()
'''

with open("scraper.py", "w", encoding="utf-8") as file:
    file.write(scraper_code)

print("scraper.py created successfully.")

scraper.py created successfully.


In [5]:
!python scraper.py

Portal opened.
Login manually in the browser.
After login, go to Attendance / Marks page.
You have 120 seconds.
Saved portal_after_login.png
Saved portal_after_login.html


In [12]:
scraper_code = r'''
import time
import re
from pathlib import Path
from playwright.sync_api import sync_playwright

PORTAL_URL = "https://springzabdesk.szabist-isb.edu.pk/"
BASE_URL = "https://springzabdesk.szabist-isb.edu.pk"

OUTPUT_DIR = Path("portal_captures")
OUTPUT_DIR.mkdir(exist_ok=True)


def save_page(page, filename_prefix):
    html_path = OUTPUT_DIR / f"{filename_prefix}.html"
    png_path = OUTPUT_DIR / f"{filename_prefix}.png"

    with open(html_path, "w", encoding="utf-8") as file:
        file.write(page.content())

    page.screenshot(path=str(png_path), full_page=True)

    print(f"Saved {html_path}")
    print(f"Saved {png_path}")


def clean_filename(text):
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    return text.strip("_").lower()


def get_link_href(page, text_value):
    locator = page.locator(f"a:has-text('{text_value}')").first
    href = locator.get_attribute("href")

    if not href:
        raise ValueError(f"Could not find href for: {text_value}")

    if href.startswith("/"):
        href = BASE_URL + href

    return href


def get_course_links(page):
    courses = []

    links = page.locator("a").all()

    for link in links:
        text = link.inner_text().strip()
        href = link.get_attribute("href")

        if href and "chkSubmit" in href:
            courses.append({
                "course_name": text,
                "href": href
            })

    return courses


def capture_course_details(page, main_url, page_type):
    print(f"\nOpening {page_type} main page...")
    page.goto(main_url)
    time.sleep(5)

    save_page(page, f"{page_type}_main_page")

    courses = get_course_links(page)
    print(f"Found {len(courses)} courses on {page_type} page.")

    for index, course in enumerate(courses, start=1):
        course_name = course["course_name"]
        safe_name = clean_filename(course_name)

        print(f"\n{page_type.upper()} {index}: {course_name}")

        page.goto(main_url)
        time.sleep(2)

        course_locator = page.locator(f"a:has-text('{course_name}')").first
        course_locator.click()

        time.sleep(5)

        save_page(page, f"{page_type}_{index}_{safe_name}")


def main():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page(viewport={"width": 1400, "height": 900})

        page.goto(PORTAL_URL)
        print("Portal opened.")

        print("\nLOGIN MANUALLY.")
        print("After login, wait. Do not close the browser.")
        print("You have 60 seconds.\n")
        time.sleep(60)

        save_page(page, "logged_in_homepage")

        attendance_url = get_link_href(page, "View Attendance")
        results_url = get_link_href(page, "Current Semester Results")

        print("\nAttendance URL:", attendance_url)
        print("Results URL:", results_url)

        capture_course_details(page, attendance_url, "attendance")
        capture_course_details(page, results_url, "results")

        print("\nDone. All detail pages saved inside portal_captures folder.")
        browser.close()


if __name__ == "__main__":
    main()
'''

with open("scraper.py", "w", encoding="utf-8") as file:
    file.write(scraper_code)

print("scraper.py updated successfully.")

scraper.py updated successfully.


In [13]:
!python scraper.py

Portal opened.

LOGIN MANUALLY.
After login, wait. Do not close the browser.
You have 60 seconds.

Saved portal_captures\logged_in_homepage.html
Saved portal_captures\logged_in_homepage.png

Attendance URL: https://springzabdesk.szabist-isb.edu.pk/Student/QryCourseAttendance.asp?OptionName=View Attendance&sid=577339771
Results URL: https://springzabdesk.szabist-isb.edu.pk/Student/QryCourseRecapSheet.asp?OptionName=Current Semester Results&sid=577339771

Opening attendance main page...
Saved portal_captures\attendance_main_page.html
Saved portal_captures\attendance_main_page.png
Found 8 courses on attendance page.

ATTENDANCE 1: CSC 4102 Professional Practices
Saved portal_captures\attendance_1_csc_4102_professional_practices.html
Saved portal_captures\attendance_1_csc_4102_professional_practices.png

ATTENDANCE 2: SEC 4516 Artificial Intelligence
Saved portal_captures\attendance_2_sec_4516_artificial_intelligence.html
Saved portal_captures\attendance_2_sec_4516_artificial_intelligence.

In [14]:
parser_code = r'''
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd

CAPTURE_DIR = Path("portal_captures")

def clean(text):
    return " ".join(text.replace("\xa0", " ").split())

def extract_course_info(soup):
    course = ""
    instructor = ""
    program = ""
    section = ""

    rows = soup.find_all("tr")

    for row in rows:
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) >= 4 and "Program:" in cells[0]:
            program = cells[1]
            section = cells[3]

        if len(cells) >= 2 and "Course:" in cells[0]:
            course = cells[1]

        if len(cells) >= 2 and "Instructor:" in cells[0]:
            instructor = cells[1]

    return course, instructor, program, section

def parse_attendance_file(file_path):
    html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    course, instructor, program, section = extract_course_info(soup)

    total_lectures = 0
    present = 0
    absent = 0
    late = 0

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all("td")]

        if len(cells) == 3 and cells[0].isdigit():
            total_lectures += 1
            status = cells[2].lower()

            if status == "present":
                present += 1
            elif status == "absent":
                absent += 1
            elif status == "late":
                late += 1

    attendance_percentage = 0

    if total_lectures > 0:
        attendance_percentage = round(((present + late) / total_lectures) * 100, 2)

    return {
        "subject": course,
        "instructor": instructor,
        "program": program,
        "section": section,
        "total_lectures": total_lectures,
        "total_present": present,
        "total_absent": absent,
        "total_late": late,
        "attendance_percentage": attendance_percentage
    }

def main():
    attendance_files = sorted(CAPTURE_DIR.glob("attendance_[0-9]*_*.html"))

    if not attendance_files:
        print("No attendance detail files found.")
        return

    rows = []

    for file_path in attendance_files:
        rows.append(parse_attendance_file(file_path))

    df = pd.DataFrame(rows)

    df["attendance_risk"] = df["attendance_percentage"].apply(
        lambda x: "High Risk" if x < 75 else "Warning" if x < 80 else "Safe"
    )

    df.to_csv("scraped_attendance_summary.csv", index=False)

    print("Saved scraped_attendance_summary.csv")
    print(df)

if __name__ == "__main__":
    main()
'''

with open("parse_attendance.py", "w", encoding="utf-8") as file:
    file.write(parser_code)

print("parse_attendance.py created successfully.")

parse_attendance.py created successfully.


In [15]:
!python parse_attendance.py

Saved scraped_attendance_summary.csv
                                      subject  ... attendance_risk
0                      Professional Practices  ...            Safe
1                     Artificial Intelligence  ...       High Risk
2      Formal Methods in Software Engineering  ...       High Risk
3                             Web Engineering  ...            Safe
4                        Information Security  ...            Safe
5                     Teachings of Holy Quran  ...       High Risk
6       Software Construction and Development  ...            Safe
7  Lab: Software Construction and Development  ...         Warning

[8 rows x 10 columns]


In [16]:
parser_code = r'''
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re

CAPTURE_DIR = Path("portal_captures")

def clean(text):
    return " ".join(text.replace("\xa0", " ").split())

def to_number(value):
    value = clean(str(value))

    if value.lower() == "not entered":
        return 0

    try:
        return float(value)
    except:
        return 0

def extract_course_info(soup):
    course = ""
    instructor = ""
    program = ""
    section = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) >= 4 and "Program:" in cells[0]:
            program = cells[1]
            section = cells[3]

        if len(cells) >= 2 and "Course:" in cells[0]:
            course = cells[1]

        if len(cells) >= 2 and "Instructor:" in cells[0]:
            instructor = cells[1]

    return course, instructor, program, section

def parse_result_file(file_path):
    html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    course, instructor, program, section = extract_course_info(soup)

    quiz_marks = 0
    assignment_marks = 0
    mid_marks = 0
    final_marks = 0
    total_marks = 0
    total_percentage = 0
    grade = "-"
    reason = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) == 3:
            head = cells[0]
            obtained = cells[2]

            if head.startswith("Quiz ("):
                quiz_marks = to_number(obtained)

            elif head.startswith("Assignment ("):
                assignment_marks = to_number(obtained)

            elif head.startswith("Mid Term Paper ("):
                mid_marks = to_number(obtained)

            elif head.startswith("Final Paper ("):
                final_marks = to_number(obtained)

            elif head == "Total Marks":
                match = re.search(r"([\d.]+)\s*/\s*100\s*\((\d+)%\)", obtained)
                if match:
                    total_marks = float(match.group(1))
                    total_percentage = float(match.group(2))

            elif head == "Grade":
                grade = obtained

            elif head == "Reason":
                reason = obtained

    return {
        "subject": course,
        "program": program,
        "section": section,
        "instructor": instructor,
        "quiz_marks": quiz_marks,
        "assignment_marks": assignment_marks,
        "mid_marks": mid_marks,
        "final_marks": final_marks,
        "total_obtained_marks": total_marks,
        "current_marks_percentage": total_percentage,
        "grade": grade,
        "reason": reason
    }

def main():
    result_files = sorted(CAPTURE_DIR.glob("results_[0-9]*_*.html"))

    if not result_files:
        print("No result detail files found.")
        return

    rows = []

    for file_path in result_files:
        rows.append(parse_result_file(file_path))

    df = pd.DataFrame(rows)

    df["marks_risk"] = df["current_marks_percentage"].apply(
        lambda x: "High Risk" if x < 50 else "Warning" if x < 65 else "Safe"
    )

    df.to_csv("scraped_marks_summary.csv", index=False)

    print("Saved scraped_marks_summary.csv")
    print(df)

if __name__ == "__main__":
    main()
'''

with open("parse_marks.py", "w", encoding="utf-8") as file:
    file.write(parser_code)

print("parse_marks.py created successfully.")

parse_marks.py created successfully.


In [17]:
!python parse_marks.py

Saved scraped_marks_summary.csv
                                      subject program  ... reason marks_risk
0                      Professional Practices  BSSE-6  ...         High Risk
1                     Artificial Intelligence  BSSE-6  ...         High Risk
2      Formal Methods in Software Engineering  BSSE-6  ...         High Risk
3                             Web Engineering  BSSE-6  ...         High Risk
4                        Information Security  BSSE-6  ...         High Risk
5                     Teachings of Holy Quran  BSSE-6  ...         High Risk
6       Software Construction and Development  BSSE-5  ...         High Risk
7  Lab: Software Construction and Development  BSSE-5  ...         High Risk

[8 rows x 13 columns]


In [18]:
merge_code = r'''
import pandas as pd

attendance = pd.read_csv("scraped_attendance_summary.csv")
marks = pd.read_csv("scraped_marks_summary.csv")

final_df = pd.merge(
    attendance,
    marks,
    on=["subject", "program", "section", "instructor"],
    how="outer"
)

def final_risk(row):
    attendance = row.get("attendance_percentage", 0)
    marks = row.get("current_marks_percentage", 0)
    reason = str(row.get("reason", "")).lower()

    if attendance < 75 or marks < 50 or "short attendance" in reason:
        return "High Risk"

    if attendance < 80 or marks < 65:
        return "Warning"

    return "Safe"

def recommendation(row):
    recs = []

    if row["attendance_percentage"] < 75:
        recs.append("Attendance is below 75%. Attend every upcoming class.")

    elif row["attendance_percentage"] < 80:
        recs.append("Attendance is close to danger zone. Avoid absents.")

    if row["current_marks_percentage"] < 50:
        recs.append("Marks are weak. Focus strongly before finals.")

    elif row["current_marks_percentage"] < 65:
        recs.append("Marks are average. Improve quizzes, assignments, and final prep.")

    if "short attendance" in str(row.get("reason", "")).lower():
        recs.append("Portal shows short attendance issue. Confirm with exam/department office.")

    if row["final_marks"] == 0:
        recs.append("Final marks are not entered yet, so current result is incomplete.")

    if not recs:
        recs.append("Subject looks stable. Maintain performance.")

    return " ".join(recs)

final_df["final_risk_status"] = final_df.apply(final_risk, axis=1)
final_df["recommendation"] = final_df.apply(recommendation, axis=1)

final_df.to_csv("final_scraped_academic_dashboard.csv", index=False)

print("Saved final_scraped_academic_dashboard.csv")
print(final_df)
'''

with open("merge_dashboard.py", "w", encoding="utf-8") as file:
    file.write(merge_code)

print("merge_dashboard.py created successfully.")

merge_dashboard.py created successfully.


In [19]:
!python merge_dashboard.py

Saved final_scraped_academic_dashboard.csv
                                      subject  ...                                     recommendation
0                     Artificial Intelligence  ...  Attendance is below 75%. Attend every upcoming...
1      Formal Methods in Software Engineering  ...  Attendance is below 75%. Attend every upcoming...
2                        Information Security  ...  Marks are weak. Focus strongly before finals. ...
3  Lab: Software Construction and Development  ...  Attendance is close to danger zone. Avoid abse...
4                      Professional Practices  ...  Marks are weak. Focus strongly before finals. ...
5       Software Construction and Development  ...  Marks are weak. Focus strongly before finals. ...
6                     Teachings of Holy Quran  ...  Attendance is below 75%. Attend every upcoming...
7                             Web Engineering  ...  Marks are weak. Focus strongly before finals. ...

[8 rows x 21 columns]


In [20]:
parser_code = r'''
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re

CAPTURE_DIR = Path("portal_captures")

def clean(text):
    return " ".join(str(text).replace("\xa0", " ").split())

def to_number(value):
    value = clean(value)

    if value.lower() in ["not entered", "-", ""]:
        return 0

    try:
        return float(value)
    except:
        return 0

def extract_total_marks(value):
    value = clean(value)
    match = re.search(r"([\d.]+)\s*/\s*100\s*\((\d+)%\)", value)

    if match:
        return float(match.group(1)), float(match.group(2))

    return 0, 0

def extract_course_info(soup):
    course = ""
    instructor = ""
    program = ""
    section = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) >= 4 and "Program:" in cells[0]:
            program = cells[1]
            section = cells[3]

        if len(cells) >= 2 and "Course:" in cells[0]:
            course = cells[1]

        if len(cells) >= 2 and "Instructor:" in cells[0]:
            instructor = cells[1]

    return course, instructor, program, section

def parse_result_file(file_path):
    html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    course, instructor, program, section = extract_course_info(soup)

    quiz_marks = 0
    assignment_marks = 0
    mid_marks = 0
    final_marks = 0
    total_marks = 0
    total_percentage = 0
    grade = "-"
    reason = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) < 2:
            continue

        head = cells[0]
        obtained = cells[-1]

        if head.startswith("Quiz ("):
            quiz_marks = to_number(obtained)

        elif head.startswith("Assignment ("):
            assignment_marks = to_number(obtained)

        elif head.startswith("Mid Term Paper ("):
            mid_marks = to_number(obtained)

        elif head.startswith("Final Paper ("):
            final_marks = to_number(obtained)

        elif head == "Total Marks":
            total_marks, total_percentage = extract_total_marks(obtained)

        elif head == "Grade":
            grade = obtained

        elif head == "Reason":
            reason = obtained

    return {
        "subject": course,
        "program": program,
        "section": section,
        "instructor": instructor,
        "quiz_marks": quiz_marks,
        "assignment_marks": assignment_marks,
        "mid_marks": mid_marks,
        "final_marks": final_marks,
        "total_obtained_marks": total_marks,
        "current_marks_percentage": total_percentage,
        "grade": grade,
        "reason": reason
    }

def main():
    result_files = sorted(CAPTURE_DIR.glob("results_[0-9]*_*.html"))

    if not result_files:
        print("No result detail files found.")
        return

    rows = []

    for file_path in result_files:
        rows.append(parse_result_file(file_path))

    df = pd.DataFrame(rows)

    df["marks_risk"] = df["current_marks_percentage"].apply(
        lambda x: "High Risk" if x < 50 else "Warning" if x < 65 else "Safe"
    )

    df.to_csv("scraped_marks_summary.csv", index=False)

    print("Saved scraped_marks_summary.csv")
    print(df)

if __name__ == "__main__":
    main()
'''

with open("parse_marks.py", "w", encoding="utf-8") as file:
    file.write(parser_code)

print("parse_marks.py fixed successfully.")

parse_marks.py fixed successfully.


In [21]:
!python parse_marks.py

Saved scraped_marks_summary.csv
                                      subject  ... marks_risk
0                      Professional Practices  ...  High Risk
1                     Artificial Intelligence  ...  High Risk
2      Formal Methods in Software Engineering  ...  High Risk
3                             Web Engineering  ...  High Risk
4                        Information Security  ...  High Risk
5                     Teachings of Holy Quran  ...  High Risk
6       Software Construction and Development  ...  High Risk
7  Lab: Software Construction and Development  ...    Warning

[8 rows x 13 columns]


In [22]:
!python merge_dashboard.py

Saved final_scraped_academic_dashboard.csv
                                      subject  ...                                     recommendation
0                     Artificial Intelligence  ...  Attendance is below 75%. Attend every upcoming...
1      Formal Methods in Software Engineering  ...  Attendance is below 75%. Attend every upcoming...
2                        Information Security  ...  Marks are weak. Focus strongly before finals. ...
3  Lab: Software Construction and Development  ...  Attendance is close to danger zone. Avoid abse...
4                      Professional Practices  ...  Marks are weak. Focus strongly before finals. ...
5       Software Construction and Development  ...  Marks are weak. Focus strongly before finals. ...
6                     Teachings of Holy Quran  ...  Attendance is below 75%. Attend every upcoming...
7                             Web Engineering  ...  Marks are weak. Focus strongly before finals. ...

[8 rows x 21 columns]


In [24]:
from pathlib import Path
import shutil

ROOT = Path.cwd()

folders = [
    "data/raw",
    "data/processed",
    "data/summaries",
    "scripts",
    "assets/screenshots",
]

for folder in folders:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

move_map = {
    "academic_data.csv": "data/raw/academic_data.csv",
    "final_academic_dashboard.csv": "data/processed/final_academic_dashboard.csv",
    "final_scraped_academic_dashboard.csv": "data/processed/final_scraped_academic_dashboard.csv",
    "scraped_attendance_summary.csv": "data/summaries/scraped_attendance_summary.csv",
    "scraped_marks_summary.csv": "data/summaries/scraped_marks_summary.csv",
    "scraper.py": "scripts/scraper.py",
    "parse_attendance.py": "scripts/parse_attendance.py",
    "parse_marks.py": "scripts/parse_marks.py",
    "merge_dashboard.py": "scripts/merge_dashboard.py",
    "portal_after_login.html": "data/raw/portal_after_login.html",
    "portal_after_login.htm": "data/raw/portal_after_login.htm",
    "portal_after_login.png": "assets/screenshots/portal_after_login.png",
}

for source, destination in move_map.items():
    source_path = ROOT / source
    destination_path = ROOT / destination

    if source_path.exists():
        if destination_path.exists():
            destination_path.unlink()
        shutil.move(str(source_path), str(destination_path))
        print(f"Moved: {source} -> {destination}")
    else:
        print(f"Skipped missing: {source}")

portal_captures_source = ROOT / "portal_captures"
portal_captures_destination = ROOT / "data/raw/portal_captures"

if portal_captures_source.exists():
    if portal_captures_destination.exists():
        shutil.rmtree(portal_captures_destination)
    shutil.move(str(portal_captures_source), str(portal_captures_destination))
    print("Moved: portal_captures -> data/raw/portal_captures")
else:
    print("Skipped missing: portal_captures")

gitignore_content = "\n".join([
    ".env",
    "__pycache__/",
    ".ipynb_checkpoints/",
    "*.pkl",
    "data/raw/portal_after_login.*",
    "data/raw/portal_captures/",
    "assets/screenshots/",
])

(ROOT / ".gitignore").write_text(gitignore_content, encoding="utf-8")

readme_lines = [
    "# SZABIST Academic Performance Dashboard",
    "",
    "## Project Overview",
    "",
    "This project analyzes academic performance using attendance and marks data from the SZABIST ZABDESK student portal.",
    "",
    "The dashboard tracks subject-wise attendance, present/absent counts, quiz marks, assignment marks, midterm marks, final marks, total marks, grade status, and academic risk.",
    "",
    "## Main Features",
    "",
    "- Scrapes academic portal pages using Playwright",
    "- Extracts subject-wise attendance data",
    "- Extracts quiz, assignment, midterm, final, total marks, grade, and result reason",
    "- Calculates attendance percentage",
    "- Detects academic risk",
    "- Generates CSV summaries",
    "- Builds a Jupyter Notebook dashboard",
    "- Keeps private scraped HTML files out of GitHub",
    "",
    "## Project Structure",
    "",
    "szabist-academic-dashboard/",
    "├── README.md",
    "├── .gitignore",
    "├── SZABIST_Academic_Dashboard.ipynb",
    "├── scraper.ipynb",
    "├── data/",
    "│   ├── raw/",
    "│   ├── processed/",
    "│   └── summaries/",
    "├── scripts/",
    "└── assets/",
    "",
    "## Technologies Used",
    "",
    "- Python",
    "- Jupyter Notebook",
    "- Pandas",
    "- BeautifulSoup",
    "- Playwright",
    "- Matplotlib",
    "",
    "## Important Files",
    "",
    "| File | Purpose |",
    "|---|---|",
    "| SZABIST_Academic_Dashboard.ipynb | Main analysis and dashboard notebook |",
    "| scraper.ipynb | Setup, scraping, parsing, and file management notebook |",
    "| scripts/scraper.py | Opens portal and captures attendance/result pages |",
    "| scripts/parse_attendance.py | Extracts attendance summary |",
    "| scripts/parse_marks.py | Extracts marks summary |",
    "| scripts/merge_dashboard.py | Merges attendance and marks into final dashboard CSV |",
    "| data/processed/final_scraped_academic_dashboard.csv | Final real scraped dashboard data |",
    "",
    "## Final Dataset",
    "",
    "Main output file:",
    "",
    "data/processed/final_scraped_academic_dashboard.csv",
    "",
    "## Privacy Warning",
    "",
    "Do not upload these files/folders to GitHub:",
    "",
    "- .env",
    "- data/raw/portal_after_login.*",
    "- data/raw/portal_captures/",
    "- assets/screenshots/",
    "",
    "These may contain private academic or personal portal data.",
    "",
    "## How to Use",
    "",
    "Open SZABIST_Academic_Dashboard.ipynb and load:",
    "",
    "import pandas as pd",
    "df = pd.read_csv('data/processed/final_scraped_academic_dashboard.csv')",
    "df",
]

(ROOT / "README.md").write_text("\n".join(readme_lines), encoding="utf-8")

print("Project folder sorted successfully.")
print("README.md updated.")
print(".gitignore updated.")

Moved: academic_data.csv -> data/raw/academic_data.csv
Moved: final_academic_dashboard.csv -> data/processed/final_academic_dashboard.csv
Moved: final_scraped_academic_dashboard.csv -> data/processed/final_scraped_academic_dashboard.csv
Moved: scraped_attendance_summary.csv -> data/summaries/scraped_attendance_summary.csv
Moved: scraped_marks_summary.csv -> data/summaries/scraped_marks_summary.csv
Moved: scraper.py -> scripts/scraper.py
Moved: parse_attendance.py -> scripts/parse_attendance.py
Moved: parse_marks.py -> scripts/parse_marks.py
Moved: merge_dashboard.py -> scripts/merge_dashboard.py
Moved: portal_after_login.html -> data/raw/portal_after_login.html
Skipped missing: portal_after_login.htm
Moved: portal_after_login.png -> assets/screenshots/portal_after_login.png
Moved: portal_captures -> data/raw/portal_captures
Project folder sorted successfully.
README.md updated.
.gitignore updated.


In [25]:
import shutil
from pathlib import Path

project_folder = Path.cwd()
zip_name = "gradescope_project"

shutil.make_archive(zip_name, "zip", project_folder)

print(f"Created: {zip_name}.zip")

Created: gradescope_project.zip


In [26]:
!pip install streamlit pandas numpy matplotlib plotly beautifulsoup4 playwright python-dotenv
!python -m playwright install

Defaulting to user installation because normal site-packages is not writeable
  Using cached pyarrow-24.0.0-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached httptools-0.8.0-cp314-cp314-win_amd64.whl.metadata (3.7 kB)
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.2 MB 2.3 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/9.2 MB 3.7 MB/s eta 0:00:03
   ----------------- ---------------------- 3.9/9.2 MB 5.9 MB/s eta 0:00:01
   -------------------------------------- - 8.9/9.2 MB 10.0 MB/s eta 0:00:01
   -------------------------------------- - 8.9/9.2 MB 10.0 MB/s eta 0:00:01
   -------------------------------------- - 8.9/9.2 MB 10.0 MB/s eta 0:00:01
   -------------------------------------- - 8.9/9.2 MB 10.0 MB/s eta 0:00:01
   ---------------------------

In [27]:
from pathlib import Path

requirements = """
streamlit
pandas
numpy
matplotlib
plotly
beautifulsoup4
playwright
python-dotenv
""".strip()

Path("requirements.txt").write_text(requirements, encoding="utf-8")

print("requirements.txt created.")

requirements.txt created.


In [28]:
from pathlib import Path

Path("scripts").mkdir(exist_ok=True)

portal_scraper_code = r'''
import time
import re
from pathlib import Path
from playwright.sync_api import sync_playwright

PORTAL_URL = "https://springzabdesk.szabist-isb.edu.pk/"
BASE_URL = "https://springzabdesk.szabist-isb.edu.pk"

ROOT = Path.cwd()
CAPTURE_DIR = ROOT / "data" / "raw" / "portal_captures"
CAPTURE_DIR.mkdir(parents=True, exist_ok=True)


def clean_filename(text):
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    return text.strip("_").lower()


def save_page(page, filename_prefix):
    html_path = CAPTURE_DIR / f"{filename_prefix}.html"
    png_path = CAPTURE_DIR / f"{filename_prefix}.png"

    html_path.write_text(page.content(), encoding="utf-8")
    page.screenshot(path=str(png_path), full_page=True)

    print(f"Saved {html_path}")
    print(f"Saved {png_path}")


def get_link_href(page, text_value):
    locator = page.locator(f"a:has-text('{text_value}')").first
    href = locator.get_attribute("href")

    if not href:
        raise ValueError(f"Could not find href for: {text_value}")

    if href.startswith("/"):
        href = BASE_URL + href

    return href


def get_course_links(page):
    courses = []

    links = page.locator("a").all()

    for link in links:
        text = link.inner_text().strip()
        href = link.get_attribute("href")

        if href and "chkSubmit" in href and text:
            courses.append({
                "course_name": text,
                "href": href
            })

    return courses


def capture_course_details(page, main_url, page_type):
    print(f"Opening {page_type} main page...")
    page.goto(main_url)
    time.sleep(4)

    save_page(page, f"{page_type}_main_page")

    courses = get_course_links(page)
    print(f"Found {len(courses)} courses on {page_type} page.")

    for index, course in enumerate(courses, start=1):
        course_name = course["course_name"]
        safe_name = clean_filename(course_name)

        print(f"{page_type.upper()} {index}: {course_name}")

        page.goto(main_url)
        time.sleep(2)

        course_locator = page.locator(f"a:has-text('{course_name}')").first
        course_locator.click()

        time.sleep(4)

        save_page(page, f"{page_type}_{index}_{safe_name}")


def run_scraper():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page(viewport={"width": 1400, "height": 900})

        page.goto(PORTAL_URL)

        print("")
        print("GradeScope scraper started.")
        print("Login manually in the opened browser.")
        print("After login, wait. The scraper will continue automatically.")
        print("Waiting 70 seconds...")
        print("")

        time.sleep(70)

        save_page(page, "logged_in_homepage")

        attendance_url = get_link_href(page, "View Attendance")
        results_url = get_link_href(page, "Current Semester Results")

        print("Attendance URL:", attendance_url)
        print("Results URL:", results_url)

        capture_course_details(page, attendance_url, "attendance")
        capture_course_details(page, results_url, "results")

        browser.close()

        print("Scraping complete.")


if __name__ == "__main__":
    run_scraper()
'''

Path("scripts/portal_scraper.py").write_text(portal_scraper_code, encoding="utf-8")

print("scripts/portal_scraper.py created.")

scripts/portal_scraper.py created.


In [29]:
from pathlib import Path

attendance_parser_code = r'''
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd

ROOT = Path.cwd()
CAPTURE_DIR = ROOT / "data" / "raw" / "portal_captures"
OUTPUT_PATH = ROOT / "data" / "summaries" / "scraped_attendance_summary.csv"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def clean(text):
    return " ".join(str(text).replace("\xa0", " ").split())


def extract_course_info(soup):
    course = ""
    instructor = ""
    program = ""
    section = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) >= 4 and "Program:" in cells[0]:
            program = cells[1]
            section = cells[3]

        if len(cells) >= 2 and "Course:" in cells[0]:
            course = cells[1]

        if len(cells) >= 2 and "Instructor:" in cells[0]:
            instructor = cells[1]

    return course, instructor, program, section


def parse_attendance_file(file_path):
    html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    course, instructor, program, section = extract_course_info(soup)

    total_lectures = 0
    present = 0
    absent = 0
    late = 0

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all("td")]

        if len(cells) == 3 and cells[0].isdigit():
            total_lectures += 1
            status = cells[2].lower()

            if status == "present":
                present += 1
            elif status == "absent":
                absent += 1
            elif status == "late":
                late += 1

    attendance_percentage = 0

    if total_lectures > 0:
        attendance_percentage = round(((present + late) / total_lectures) * 100, 2)

    return {
        "subject": course,
        "instructor": instructor,
        "program": program,
        "section": section,
        "total_lectures": total_lectures,
        "total_present": present,
        "total_absent": absent,
        "total_late": late,
        "attendance_percentage": attendance_percentage
    }


def run_attendance_parser():
    attendance_files = sorted(CAPTURE_DIR.glob("attendance_[0-9]*_*.html"))

    if not attendance_files:
        raise FileNotFoundError("No attendance detail files found. Run scraper first.")

    rows = [parse_attendance_file(file_path) for file_path in attendance_files]

    df = pd.DataFrame(rows)

    df["attendance_risk"] = df["attendance_percentage"].apply(
        lambda x: "High Risk" if x < 80 else "Safe"
    )

    df.to_csv(OUTPUT_PATH, index=False)

    print(f"Saved {OUTPUT_PATH}")
    return df


if __name__ == "__main__":
    run_attendance_parser()
'''

Path("scripts/parse_attendance.py").write_text(attendance_parser_code, encoding="utf-8")

print("scripts/parse_attendance.py created.")

scripts/parse_attendance.py created.


In [30]:
from pathlib import Path

marks_parser_code = r'''
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re

ROOT = Path.cwd()
CAPTURE_DIR = ROOT / "data" / "raw" / "portal_captures"
OUTPUT_PATH = ROOT / "data" / "summaries" / "scraped_marks_summary.csv"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def clean(text):
    return " ".join(str(text).replace("\xa0", " ").split())


def to_number(value):
    value = clean(value)

    if value.lower() in ["not entered", "-", ""]:
        return 0

    try:
        return float(value)
    except:
        return 0


def extract_total_marks(value):
    value = clean(value)
    match = re.search(r"([\d.]+)\s*/\s*100\s*\((\d+)%\)", value)

    if match:
        return float(match.group(1)), float(match.group(2))

    return 0, 0


def extract_course_info(soup):
    course = ""
    instructor = ""
    program = ""
    section = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) >= 4 and "Program:" in cells[0]:
            program = cells[1]
            section = cells[3]

        if len(cells) >= 2 and "Course:" in cells[0]:
            course = cells[1]

        if len(cells) >= 2 and "Instructor:" in cells[0]:
            instructor = cells[1]

    return course, instructor, program, section


def parse_result_file(file_path):
    html = file_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")

    course, instructor, program, section = extract_course_info(soup)

    quiz_marks = 0
    assignment_marks = 0
    mid_marks = 0
    final_marks = 0
    total_marks = 0
    total_percentage = 0
    grade = "-"
    reason = ""

    for row in soup.find_all("tr"):
        cells = [clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]

        if len(cells) < 2:
            continue

        head = cells[0]
        obtained = cells[-1]

        if head.startswith("Quiz ("):
            quiz_marks = to_number(obtained)

        elif head.startswith("Assignment ("):
            assignment_marks = to_number(obtained)

        elif head.startswith("Mid Term Paper ("):
            mid_marks = to_number(obtained)

        elif head.startswith("Final Paper ("):
            final_marks = to_number(obtained)

        elif head == "Total Marks":
            total_marks, total_percentage = extract_total_marks(obtained)

        elif head == "Grade":
            grade = obtained

        elif head == "Reason":
            reason = obtained

    return {
        "subject": course,
        "program": program,
        "section": section,
        "instructor": instructor,
        "quiz_marks": quiz_marks,
        "assignment_marks": assignment_marks,
        "mid_marks": mid_marks,
        "final_marks": final_marks,
        "total_obtained_marks": total_marks,
        "current_marks_percentage": total_percentage,
        "grade": grade,
        "reason": reason
    }


def run_marks_parser():
    result_files = sorted(CAPTURE_DIR.glob("results_[0-9]*_*.html"))

    if not result_files:
        raise FileNotFoundError("No result detail files found. Run scraper first.")

    rows = [parse_result_file(file_path) for file_path in result_files]

    df = pd.DataFrame(rows)

    df["marks_risk"] = df["current_marks_percentage"].apply(
        lambda x: "High Risk" if x < 55 else "Safe"
    )

    df.to_csv(OUTPUT_PATH, index=False)

    print(f"Saved {OUTPUT_PATH}")
    return df


if __name__ == "__main__":
    run_marks_parser()
'''

Path("scripts/parse_marks.py").write_text(marks_parser_code, encoding="utf-8")

print("scripts/parse_marks.py created.")

scripts/parse_marks.py created.


In [31]:
from pathlib import Path

merge_code = r'''
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

ATTENDANCE_PATH = ROOT / "data" / "summaries" / "scraped_attendance_summary.csv"
MARKS_PATH = ROOT / "data" / "summaries" / "scraped_marks_summary.csv"
OUTPUT_PATH = ROOT / "data" / "processed" / "gradescope_final_dashboard.csv"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def final_risk(row):
    attendance = row.get("attendance_percentage", 0)
    marks = row.get("current_marks_percentage", 0)
    reason = str(row.get("reason", "")).lower()

    if attendance < 80 or marks < 55 or "short attendance" in reason:
        return "High Risk"

    return "Safe"


def recommendation(row):
    recs = []

    if row["attendance_percentage"] < 80:
        recs.append("Attendance is below the safe 80% level. Attend every upcoming class.")

    if row["current_marks_percentage"] < 55:
        recs.append("Marks are below passing level. Prioritize this subject before finals.")

    if "short attendance" in str(row.get("reason", "")).lower():
        recs.append("Portal shows short attendance issue. Confirm with the department.")

    if row["final_marks"] == 0:
        recs.append("Final marks are not entered yet, so current result is incomplete.")

    if not recs:
        recs.append("Subject is currently safe. Maintain attendance and marks.")

    return " ".join(recs)


def run_merge():
    if not ATTENDANCE_PATH.exists():
        raise FileNotFoundError("Missing scraped_attendance_summary.csv. Run attendance parser first.")

    if not MARKS_PATH.exists():
        raise FileNotFoundError("Missing scraped_marks_summary.csv. Run marks parser first.")

    attendance = pd.read_csv(ATTENDANCE_PATH)
    marks = pd.read_csv(MARKS_PATH)

    final_df = pd.merge(
        attendance,
        marks,
        on=["subject", "program", "section", "instructor"],
        how="outer"
    )

    numeric_columns = [
        "attendance_percentage",
        "total_present",
        "total_absent",
        "total_late",
        "quiz_marks",
        "assignment_marks",
        "mid_marks",
        "final_marks",
        "total_obtained_marks",
        "current_marks_percentage"
    ]

    for col in numeric_columns:
        if col in final_df.columns:
            final_df[col] = pd.to_numeric(final_df[col], errors="coerce").fillna(0)

    final_df["final_risk_status"] = final_df.apply(final_risk, axis=1)
    final_df["recommendation"] = final_df.apply(recommendation, axis=1)

    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"Saved {OUTPUT_PATH}")
    return final_df


if __name__ == "__main__":
    run_merge()
'''

Path("scripts/merge_dashboard.py").write_text(merge_code, encoding="utf-8")

print("scripts/merge_dashboard.py created.")

scripts/merge_dashboard.py created.


In [32]:
from pathlib import Path

app_code = r'''
import subprocess
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "processed" / "gradescope_final_dashboard.csv"

st.set_page_config(
    page_title="GradeScope Portal",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

CUSTOM_CSS = """
<style>
:root {
    --bg: #0f172a;
    --card: #111827;
    --muted: #94a3b8;
    --text: #f8fafc;
    --red: #ef4444;
    --orange: #f97316;
    --green: #22c55e;
    --blue: #38bdf8;
}

.stApp {
    background: linear-gradient(135deg, #020617 0%, #0f172a 45%, #111827 100%);
    color: var(--text);
}

.main-title {
    font-size: 48px;
    font-weight: 900;
    letter-spacing: -1px;
    margin-bottom: 0px;
}

.subtitle {
    font-size: 18px;
    color: #cbd5e1;
    margin-bottom: 24px;
}

.glass-card {
    padding: 22px;
    border-radius: 22px;
    background: rgba(15, 23, 42, 0.78);
    border: 1px solid rgba(148, 163, 184, 0.18);
    box-shadow: 0 20px 60px rgba(0,0,0,0.35);
}

.metric-card {
    padding: 18px;
    border-radius: 20px;
    background: rgba(30, 41, 59, 0.85);
    border: 1px solid rgba(148, 163, 184, 0.18);
    text-align: center;
}

.metric-value {
    font-size: 34px;
    font-weight: 900;
}

.metric-label {
    color: #cbd5e1;
    font-size: 14px;
}

.high-risk {
    color: #fecaca;
}

.safe-risk {
    color: #bbf7d0;
}

.warning-box {
    padding: 15px;
    border-radius: 16px;
    background: rgba(239, 68, 68, 0.12);
    border: 1px solid rgba(239, 68, 68, 0.35);
}

.success-box {
    padding: 15px;
    border-radius: 16px;
    background: rgba(34, 197, 94, 0.12);
    border: 1px solid rgba(34, 197, 94, 0.35);
}
</style>
"""

st.markdown(CUSTOM_CSS, unsafe_allow_html=True)


def run_command(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        shell=True
    )

    output_box = st.empty()
    logs = ""

    for line in process.stdout:
        logs += line
        output_box.code(logs)

    process.wait()

    if process.returncode != 0:
        st.error("Command failed.")
    else:
        st.success("Done.")


def load_data():
    if DATA_PATH.exists():
        return pd.read_csv(DATA_PATH)
    return pd.DataFrame()


def prepare_df(df):
    if df.empty:
        return df

    numeric_columns = [
        "attendance_percentage",
        "total_present",
        "total_absent",
        "total_late",
        "quiz_marks",
        "assignment_marks",
        "mid_marks",
        "final_marks",
        "total_obtained_marks",
        "current_marks_percentage"
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["subject_short"] = df["subject"].astype(str)

    replacements = {
        "Software Construction and Development": "SCD",
        "Lab: Software Construction and Development": "SCD Lab",
        "Formal Methods in Software Engineering": "Formal Methods",
        "Artificial Intelligence": "AI",
        "Information Security": "InfoSec",
        "Professional Practices": "Pro Practices",
        "Web Engineering": "Web Eng",
        "Teachings of Holy Quran": "Quran"
    }

    for full, short in replacements.items():
        df["subject_short"] = df["subject_short"].str.replace(full, short, regex=False)

    return df


def risk_color(status):
    if status == "High Risk":
        return "#ef4444"
    return "#22c55e"


def render_empty_state():
    st.markdown('<div class="glass-card">', unsafe_allow_html=True)
    st.subheader("No dashboard data found yet")
    st.write("Click the scraping button from the sidebar. Login manually when the ZABDesk browser opens.")
    st.markdown("</div>", unsafe_allow_html=True)


def render_kpis(df):
    total_subjects = len(df)
    high_risk = int((df["final_risk_status"] == "High Risk").sum())
    safe = int((df["final_risk_status"] == "Safe").sum())
    avg_attendance = round(df["attendance_percentage"].mean(), 2)
    avg_marks = round(df["current_marks_percentage"].mean(), 2)

    c1, c2, c3, c4, c5 = st.columns(5)

    cards = [
        (c1, total_subjects, "Subjects"),
        (c2, high_risk, "High Risk"),
        (c3, safe, "Safe"),
        (c4, f"{avg_attendance}%", "Average Attendance"),
        (c5, f"{avg_marks}%", "Average Marks")
    ]

    for col, value, label in cards:
        with col:
            st.markdown(
                f"""
                <div class="metric-card">
                    <div class="metric-value">{value}</div>
                    <div class="metric-label">{label}</div>
                </div>
                """,
                unsafe_allow_html=True
            )


def render_charts(df):
    color_map = {
        "High Risk": "#ef4444",
        "Safe": "#22c55e"
    }

    st.markdown("### Visual Dashboard")

    col1, col2 = st.columns(2)

    with col1:
        fig = px.bar(
            df,
            x="subject_short",
            y="attendance_percentage",
            color="final_risk_status",
            color_discrete_map=color_map,
            title="Attendance Health",
            text="attendance_percentage"
        )
        fig.add_hline(y=80, line_dash="dash", line_color="#facc15", annotation_text="Acceptable Attendance 80%")
        fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
        fig.update_layout(template="plotly_dark", yaxis_range=[0, 110], height=430)
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        fig = px.bar(
            df,
            x="subject_short",
            y="current_marks_percentage",
            color="final_risk_status",
            color_discrete_map=color_map,
            title="Marks Performance",
            text="current_marks_percentage"
        )
        fig.add_hline(y=55, line_dash="dash", line_color="#facc15", annotation_text="Passing Marks 55%")
        fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside")
        fig.update_layout(template="plotly_dark", yaxis_range=[0, 110], height=430)
        st.plotly_chart(fig, use_container_width=True)

    col3, col4 = st.columns(2)

    with col3:
        risk_counts = df["final_risk_status"].value_counts().reset_index()
        risk_counts.columns = ["risk", "count"]

        fig = px.pie(
            risk_counts,
            names="risk",
            values="count",
            hole=0.45,
            title="Risk Distribution",
            color="risk",
            color_discrete_map=color_map
        )
        fig.update_layout(template="plotly_dark", height=430)
        st.plotly_chart(fig, use_container_width=True)

    with col4:
        fig = px.scatter(
            df,
            x="attendance_percentage",
            y="current_marks_percentage",
            color="final_risk_status",
            color_discrete_map=color_map,
            text="subject_short",
            size="total_absent",
            title="Risk Map: Attendance vs Marks"
        )
        fig.add_vline(x=80, line_dash="dash", line_color="#facc15")
        fig.add_hline(y=55, line_dash="dash", line_color="#facc15")
        fig.update_traces(textposition="top center")
        fig.update_layout(template="plotly_dark", xaxis_range=[0, 105], yaxis_range=[0, 105], height=430)
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("### Trend-like Performance Wave")

    wave_df = df.sort_values("current_marks_percentage").copy()
    fig = px.line(
        wave_df,
        x="subject_short",
        y=["attendance_percentage", "current_marks_percentage"],
        markers=True,
        title="Attendance and Marks Wave"
    )
    fig.add_hline(y=80, line_dash="dash", line_color="#38bdf8", annotation_text="80% Attendance")
    fig.add_hline(y=55, line_dash="dash", line_color="#facc15", annotation_text="55% Passing")
    fig.update_layout(template="plotly_dark", height=460)
    st.plotly_chart(fig, use_container_width=True)


def render_table(df):
    st.markdown("### Subject Risk Table")

    view_cols = [
        "subject",
        "attendance_percentage",
        "total_present",
        "total_absent",
        "total_late",
        "quiz_marks",
        "assignment_marks",
        "mid_marks",
        "final_marks",
        "current_marks_percentage",
        "grade",
        "reason",
        "final_risk_status",
        "recommendation"
    ]

    available_cols = [col for col in view_cols if col in df.columns]

    st.dataframe(
        df[available_cols],
        use_container_width=True,
        hide_index=True
    )


def render_priority_plan(df):
    st.markdown("### Priority Action Plan")

    risky = df[df["final_risk_status"] == "High Risk"].copy()

    if risky.empty:
        st.markdown(
            '<div class="success-box">All subjects are currently safe based on 80% attendance and 55% passing marks.</div>',
            unsafe_allow_html=True
        )
        return

    risky = risky.sort_values(["attendance_percentage", "current_marks_percentage"])

    for _, row in risky.iterrows():
        st.markdown(
            f"""
            <div class="warning-box">
                <b>{row['subject']}</b><br>
                Attendance: {row['attendance_percentage']:.2f}% |
                Marks: {row['current_marks_percentage']:.0f}%<br>
                {row['recommendation']}
            </div>
            <br>
            """,
            unsafe_allow_html=True
        )


st.sidebar.title("GradeScope Controls")

st.sidebar.markdown("### Thresholds")
st.sidebar.info("Acceptable attendance: 80%\n\nPassing marks: 55%")

if st.sidebar.button("1. Open ZABDesk and scrape portal"):
    run_command(f'"{sys.executable}" scripts/portal_scraper.py')

if st.sidebar.button("2. Parse attendance"):
    run_command(f'"{sys.executable}" scripts/parse_attendance.py')

if st.sidebar.button("3. Parse marks"):
    run_command(f'"{sys.executable}" scripts/parse_marks.py')

if st.sidebar.button("4. Build dashboard dataset"):
    run_command(f'"{sys.executable}" scripts/merge_dashboard.py')

if st.sidebar.button("Run full local pipeline after scraping"):
    run_command(f'"{sys.executable}" scripts/parse_attendance.py')
    run_command(f'"{sys.executable}" scripts/parse_marks.py')
    run_command(f'"{sys.executable}" scripts/merge_dashboard.py')

st.markdown('<div class="main-title">GradeScope Portal</div>', unsafe_allow_html=True)
st.markdown('<div class="subtitle">Local academic risk dashboard with ZABDesk portal automation</div>', unsafe_allow_html=True)

df = prepare_df(load_data())

if df.empty:
    render_empty_state()
else:
    render_kpis(df)
    st.divider()
    render_charts(df)
    st.divider()
    render_priority_plan(df)
    st.divider()
    render_table(df)
'''

Path("app.py").write_text(app_code, encoding="utf-8")

print("app.py created.")

app.py created.


In [33]:
from pathlib import Path

setup_bat = r'''
@echo off
title GradeScope Setup
echo ==========================================
echo GradeScope Setup
echo ==========================================

python --version
IF ERRORLEVEL 1 (
    echo Python is not installed or not added to PATH.
    echo Install Python first from https://www.python.org/downloads/
    pause
    exit /b
)

echo.
echo Creating virtual environment...
python -m venv .venv

echo.
echo Activating virtual environment...
call .venv\Scripts\activate

echo.
echo Upgrading pip...
python -m pip install --upgrade pip

echo.
echo Installing requirements...
pip install -r requirements.txt

echo.
echo Installing Playwright browser...
python -m playwright install

echo.
echo Setup complete.
echo Run start_gradescope.bat to open the app.
pause
'''

start_bat = r'''
@echo off
title GradeScope Portal
echo ==========================================
echo Starting GradeScope Portal
echo ==========================================

IF NOT EXIST ".venv\Scripts\activate.bat" (
    echo Virtual environment not found.
    echo Run setup.bat first.
    pause
    exit /b
)

call .venv\Scripts\activate

echo Opening GradeScope in browser...
streamlit run app.py

pause
'''

Path("setup.bat").write_text(setup_bat.strip(), encoding="utf-8")
Path("start_gradescope.bat").write_text(start_bat.strip(), encoding="utf-8")

print("setup.bat created.")
print("start_gradescope.bat created.")

setup.bat created.
start_gradescope.bat created.


In [34]:
from pathlib import Path

gitignore = """
.env
.venv/
__pycache__/
.ipynb_checkpoints/

data/raw/
data/summaries/
data/processed/final_scraped_academic_dashboard.csv

assets/screenshots/

*.html
*.htm
!assets/reports/gradescope_report.html
""".strip()

Path(".gitignore").write_text(gitignore, encoding="utf-8")

print(".gitignore updated.")

.gitignore updated.


In [36]:
from pathlib import Path
import zipfile
import fnmatch

ROOT = Path.cwd()
ZIP_NAME = "GradeScope_Public_Release.zip"

# -----------------------------
# README.md
# -----------------------------

readme_lines = [
    "# GradeScope",
    "",
    "A local academic risk dashboard that turns raw ZABDesk portal data into clear attendance, marks, risk, and recommendation insights.",
    "",
    "GradeScope opens the student portal locally, lets the student log in manually, scrapes attendance and marks pages, parses the data, and displays everything in a clean Streamlit dashboard.",
    "",
    "## What this project does",
    "",
    "- Opens the ZABDesk portal through browser automation",
    "- Uses manual login, so no password is stored in the app",
    "- Scrapes subject attendance pages",
    "- Scrapes subject marks pages",
    "- Extracts attendance, absents, late counts, quizzes, assignments, mids, finals, total marks, grades, and result reasons",
    "- Builds a clean local dashboard",
    "- Shows risk using practical thresholds",
    "- Generates charts, recommendations, and processed CSV outputs",
    "",
    "## Risk rules",
    "",
    "| Metric | Safe level | Risk condition |",
    "|---|---:|---|",
    "| Attendance | 80% or above | Below 80% |",
    "| Marks | 55% or above | Below 55% |",
    "",
    "## Preview",
    "",
    "### Executive summary",
    "",
    "![KPI Summary](assets/charts/kpi_summary.png)",
    "",
    "### Attendance health",
    "",
    "![Attendance Health](assets/charts/attendance_health_pro.png)",
    "",
    "### Marks performance",
    "",
    "![Marks Performance](assets/charts/marks_performance_pro.png)",
    "",
    "### Risk distribution",
    "",
    "![Risk Distribution](assets/charts/risk_distribution_donut.png)",
    "",
    "### Attendance vs marks risk map",
    "",
    "![Risk Map](assets/charts/risk_map_pro.png)",
    "",
    "## Local app",
    "",
    "GradeScope includes a local Streamlit portal.",
    "",
    "After setup, run:",
    "",
    "```text",
    "start_gradescope.bat",
    "```",
    "",
    "The dashboard opens in your browser.",
    "",
    "## Quick start on Windows",
    "",
    "### 1. Install",
    "",
    "Double-click:",
    "",
    "```text",
    "setup.bat",
    "```",
    "",
    "This creates a virtual environment, installs Python dependencies, and installs the Playwright browser.",
    "",
    "### 2. Start",
    "",
    "Double-click:",
    "",
    "```text",
    "start_gradescope.bat",
    "```",
    "",
    "This starts the local GradeScope dashboard.",
    "",
    "### 3. Use the dashboard",
    "",
    "Inside the app:",
    "",
    "1. Click `Open ZABDesk and scrape portal`.",
    "2. Login manually when the browser opens.",
    "3. Wait for scraping to complete.",
    "4. Click `Run full local pipeline after scraping`.",
    "5. View attendance, marks, charts, risk status, and recommendations.",
    "",
    "## Manual setup",
    "",
    "```bash",
    "pip install -r requirements.txt",
    "python -m playwright install",
    "streamlit run app.py",
    "```",
    "",
    "## Project structure",
    "",
    "```text",
    "gradescope/",
    "├── app.py",
    "├── setup.bat",
    "├── start_gradescope.bat",
    "├── requirements.txt",
    "├── README.md",
    "├── SZABIST_Academic_Dashboard.ipynb",
    "├── scraper.ipynb",
    "├── scripts/",
    "│   ├── portal_scraper.py",
    "│   ├── parse_attendance.py",
    "│   ├── parse_marks.py",
    "│   └── merge_dashboard.py",
    "├── data/",
    "│   └── processed/",
    "│       └── demo_gradescope_dashboard.csv",
    "├── assets/",
    "│   ├── charts/",
    "│   └── reports/",
    "│       └── gradescope_report.html",
    "└── docs/",
    "    ├── DATA_DICTIONARY.md",
    "    └── PRIVACY_CHECKLIST.md",
    "```",
    "",
    "## Main files",
    "",
    "| File | Purpose |",
    "|---|---|",
    "| `app.py` | Local Streamlit dashboard |",
    "| `setup.bat` | One-click setup for Windows |",
    "| `start_gradescope.bat` | One-click app launcher |",
    "| `scripts/portal_scraper.py` | Opens portal and captures pages |",
    "| `scripts/parse_attendance.py` | Extracts attendance data |",
    "| `scripts/parse_marks.py` | Extracts marks data |",
    "| `scripts/merge_dashboard.py` | Builds the final dashboard dataset |",
    "| `data/processed/demo_gradescope_dashboard.csv` | Public-safe sample data |",
    "",
    "## Tech stack",
    "",
    "- Python",
    "- Streamlit",
    "- Playwright",
    "- BeautifulSoup",
    "- Pandas",
    "- NumPy",
    "- Plotly",
    "- Matplotlib",
    "- Jupyter Notebook",
    "",
    "## Privacy",
    "",
    "GradeScope is built for local personal use.",
    "",
    "Do not upload real portal captures, screenshots, login credentials, or private academic data.",
    "",
    "The public release should only include demo or sanitized data.",
    "",
    "Ignored private paths include:",
    "",
    "```text",
    ".env",
    ".venv/",
    "data/raw/",
    "data/summaries/",
    "assets/screenshots/",
    "real scraped CSV files",
    "portal HTML captures",
    "```",
    "",
    "## Current status",
    "",
    "GradeScope is functional as a local academic risk portal. It can scrape, parse, merge, visualize, and report academic performance data locally.",
]

(ROOT / "README.md").write_text("\n".join(readme_lines), encoding="utf-8")
print("README.md updated.")

# -----------------------------
# setup.bat
# -----------------------------

setup_bat = r"""
@echo off
title GradeScope Setup
echo ==========================================
echo GradeScope Setup
echo ==========================================
echo.

python --version >nul 2>&1
IF ERRORLEVEL 1 (
    echo Python is not installed or not added to PATH.
    echo Install Python from https://www.python.org/downloads/
    echo During installation, enable: Add Python to PATH
    pause
    exit /b 1
)

echo Python found.
python --version
echo.

IF NOT EXIST ".venv" (
    echo Creating virtual environment...
    python -m venv .venv
) ELSE (
    echo Virtual environment already exists.
)

echo.
echo Activating virtual environment...
call .venv\Scripts\activate.bat

echo.
echo Upgrading pip...
python -m pip install --upgrade pip

echo.
echo Installing requirements...
pip install -r requirements.txt
IF ERRORLEVEL 1 (
    echo Failed to install requirements.
    pause
    exit /b 1
)

echo.
echo Installing Playwright browser...
python -m playwright install
IF ERRORLEVEL 1 (
    echo Failed to install Playwright browsers.
    pause
    exit /b 1
)

echo.
echo ==========================================
echo Setup complete.
echo Run start_gradescope.bat to launch GradeScope.
echo ==========================================
pause
"""

(ROOT / "setup.bat").write_text(setup_bat.strip(), encoding="utf-8")
print("setup.bat updated.")

# -----------------------------
# start_gradescope.bat
# -----------------------------

start_bat = r"""
@echo off
title GradeScope Portal
echo ==========================================
echo Starting GradeScope Portal
echo ==========================================
echo.

IF NOT EXIST ".venv\Scripts\activate.bat" (
    echo Virtual environment not found.
    echo Run setup.bat first.
    pause
    exit /b 1
)

call .venv\Scripts\activate.bat

IF NOT EXIST "app.py" (
    echo app.py not found.
    echo Make sure you are running this file from the GradeScope project folder.
    pause
    exit /b 1
)

echo Opening GradeScope in your browser...
echo.
streamlit run app.py

pause
"""

(ROOT / "start_gradescope.bat").write_text(start_bat.strip(), encoding="utf-8")
print("start_gradescope.bat updated.")

# -----------------------------
# .gitignore
# -----------------------------

gitignore_lines = [
    ".env",
    ".venv/",
    "__pycache__/",
    ".ipynb_checkpoints/",
    "",
    "data/raw/",
    "data/summaries/",
    "data/processed/final_scraped_academic_dashboard.csv",
    "",
    "assets/screenshots/",
    "",
    "*.html",
    "*.htm",
    "!assets/reports/gradescope_report.html",
    "",
    "GradeScope_Public_Release.zip",
]

(ROOT / ".gitignore").write_text("\n".join(gitignore_lines), encoding="utf-8")
print(".gitignore updated.")

# -----------------------------
# Clean ZIP
# -----------------------------

exclude_patterns = [
    ".env",
    ".venv/*",
    "__pycache__/*",
    ".ipynb_checkpoints/*",
    "data/raw/*",
    "data/summaries/*",
    "data/processed/final_scraped_academic_dashboard.csv",
    "assets/screenshots/*",
    "*.htm",
    "*.html",
    "GradeScope_Public_Release.zip",
]

allow_patterns = [
    "assets/reports/gradescope_report.html",
]

def should_exclude(path):
    rel = path.as_posix()

    for allow in allow_patterns:
        if fnmatch.fnmatch(rel, allow):
            return False

    for pattern in exclude_patterns:
        if fnmatch.fnmatch(rel, pattern):
            return True

    return False

zip_path = ROOT / ZIP_NAME

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in ROOT.rglob("*"):
        if file_path.is_file():
            relative_path = file_path.relative_to(ROOT)

            if should_exclude(relative_path):
                continue

            zipf.write(file_path, relative_path)

print(f"Clean ZIP created: {ZIP_NAME}")
print("Download it from JupyterLab by right-clicking the ZIP and selecting Download.")

README.md updated.
setup.bat updated.
start_gradescope.bat updated.
.gitignore updated.
Clean ZIP created: GradeScope_Public_Release.zip
Download it from JupyterLab by right-clicking the ZIP and selecting Download.


In [3]:
from pathlib import Path

app_code = r'''
from pathlib import Path
import subprocess
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

ROOT = Path.cwd()

FINAL_DATA_PATH = ROOT / "data" / "processed" / "gradescope_final_dashboard.csv"
ATTENDANCE_PATH = ROOT / "data" / "summaries" / "scraped_attendance_summary.csv"
MARKS_PATH = ROOT / "data" / "summaries" / "scraped_marks_summary.csv"

st.set_page_config(
    page_title="GradeScope Portal",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

CUSTOM_CSS = """
<style>
:root {
    --bg: #F7FAF7;
    --card: #FFFFFF;
    --line: #E4ECE7;
    --text: #111827;
    --muted: #5B6472;
    --sage: #8FAE8E;
    --sage-dark: #6F8F72;
    --sage-soft: #E7F0E8;
    --blue: #A9D6E5;
    --blue-soft: #EAF5FA;
    --black: #101828;
    --danger: #D96C5F;
    --warning: #E7B34C;
    --shadow: 0 14px 40px rgba(16, 24, 40, 0.08);
    --radius: 22px;
}

.stApp {
    background:
        radial-gradient(circle at top left, rgba(169,214,229,0.28) 0%, rgba(169,214,229,0) 26%),
        radial-gradient(circle at top right, rgba(143,174,142,0.18) 0%, rgba(143,174,142,0) 26%),
        var(--bg);
    color: var(--text);
}

section[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #FFFFFF 0%, #F5FAF6 100%);
    border-right: 1px solid #E3ECE7;
}

.sidebar-brand {
    animation: fadeUp 0.45s ease;
    margin-bottom: 1rem;
    padding: 18px;
    border-radius: 24px;
    background: linear-gradient(135deg, #0E1116 0%, #16202C 100%);
    color: white;
    box-shadow: 0 16px 40px rgba(16, 24, 40, 0.18);
}

.sidebar-brand h1 {
    margin: 0;
    font-size: 30px;
    line-height: 1;
    letter-spacing: -0.04em;
}

.sidebar-brand p {
    margin: 8px 0 0 0;
    color: rgba(255,255,255,0.78);
    font-size: 13px;
}

.sidebar-card {
    animation: fadeUp 0.6s ease;
    background: linear-gradient(135deg, #F8FCFE 0%, #EEF6F3 100%);
    border: 1px solid #E0ECE6;
    border-radius: 20px;
    padding: 14px 16px;
    color: var(--text);
    margin-top: 10px;
    box-shadow: 0 10px 24px rgba(16, 24, 40, 0.05);
}

.sidebar-card-title {
    font-size: 12px;
    font-weight: 700;
    text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 8px;
    letter-spacing: 0.08em;
}

.sidebar-note {
    font-size: 13px;
    color: var(--muted);
    line-height: 1.5;
}

.stButton > button {
    width: 100%;
    border-radius: 18px;
    min-height: 52px;
    border: 1px solid #DCE8E1;
    background: white;
    color: var(--black);
    font-weight: 700;
    font-size: 15px;
    transition: all 0.22s ease;
    box-shadow: 0 10px 24px rgba(16, 24, 40, 0.05);
}

.stButton > button:hover {
    transform: translateY(-2px);
    border-color: #9DC1B3;
    box-shadow: 0 16px 34px rgba(16, 24, 40, 0.10);
    color: #0E1116;
}

.hero {
    animation: fadeUp 0.5s ease;
    border-radius: 30px;
    background: linear-gradient(135deg, #FFFFFF 0%, #F2F9F6 58%, #EEF7FB 100%);
    border: 1px solid #E3ECE7;
    padding: 34px;
    box-shadow: var(--shadow);
    margin-bottom: 24px;
}

.hero-badge {
    display: inline-block;
    padding: 8px 14px;
    border-radius: 999px;
    background: #E7F0E8;
    color: #47624B;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    margin-bottom: 18px;
}

.hero-title {
    font-size: 54px;
    line-height: 1;
    font-weight: 900;
    letter-spacing: -0.05em;
    color: var(--black);
    margin-bottom: 10px;
}

.hero-subtitle {
    font-size: 18px;
    color: var(--muted);
    max-width: 760px;
    line-height: 1.65;
}

.section-title {
    font-size: 28px;
    font-weight: 900;
    letter-spacing: -0.04em;
    color: var(--black);
    margin: 16px 0;
    animation: fadeUp 0.55s ease;
}

.surface {
    animation: fadeUp 0.55s ease;
    background: rgba(255,255,255,0.90);
    border: 1px solid #E4ECE7;
    border-radius: 26px;
    box-shadow: var(--shadow);
    padding: 22px;
}

.metric-wrap {
    animation: fadeUp 0.55s ease;
    background: linear-gradient(135deg, #FFFFFF 0%, #F8FCFA 100%);
    border: 1px solid #E4ECE7;
    border-radius: 24px;
    box-shadow: var(--shadow);
    padding: 20px;
    min-height: 132px;
    transition: all 0.22s ease;
}

.metric-wrap:hover {
    transform: translateY(-3px);
    box-shadow: 0 18px 38px rgba(16,24,40,0.11);
}

.metric-top {
    font-size: 12px;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    color: var(--muted);
    margin-bottom: 15px;
}

.metric-value {
    font-size: 42px;
    font-weight: 900;
    letter-spacing: -0.04em;
    color: var(--black);
    line-height: 1;
}

.metric-foot {
    font-size: 13px;
    color: var(--muted);
    margin-top: 10px;
}

.badge {
    display: inline-block;
    padding: 8px 12px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 0.04em;
}

.pill-safe {
    color: #426646;
    background: #EAF5EC;
    border: 1px solid #D3E8D7;
}

.pill-risk {
    color: #8E433A;
    background: #FDEDEA;
    border: 1px solid #F2D0CB;
}

.callout {
    animation: fadeUp 0.5s ease;
    padding: 18px;
    border-radius: 20px;
    border: 1px solid #E4ECE7;
    background: white;
    box-shadow: var(--shadow);
    margin-bottom: 14px;
}

.callout-title {
    font-size: 16px;
    font-weight: 900;
    color: var(--black);
    margin-bottom: 8px;
}

.callout-meta {
    font-size: 13px;
    color: var(--muted);
    margin-bottom: 10px;
}

.callout-body {
    font-size: 14px;
    color: var(--text);
    line-height: 1.65;
}

.sync-panel {
    animation: fadeUp 0.5s ease;
    border-radius: 26px;
    background: linear-gradient(135deg, #FFFFFF 0%, #F6FAF7 100%);
    border: 1px solid #E4ECE7;
    box-shadow: var(--shadow);
    padding: 26px;
}

.sync-step {
    border-left: 4px solid #A9D6E5;
    padding: 12px 0 12px 16px;
    margin-bottom: 12px;
    background: #FBFEFF;
    border-radius: 0 14px 14px 0;
}

@keyframes fadeUp {
    from {
        opacity: 0;
        transform: translateY(14px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}
</style>
"""

st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

if "view" not in st.session_state:
    st.session_state.view = "dashboard"


def set_view(view_name):
    st.session_state.view = view_name


def load_csv(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def prepare_df(df):
    if df.empty:
        return df

    df = df.copy()

    numeric_columns = [
        "attendance_percentage",
        "total_present",
        "total_absent",
        "total_late",
        "quiz_marks",
        "assignment_marks",
        "mid_marks",
        "final_marks",
        "total_obtained_marks",
        "current_marks_percentage"
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["subject_short"] = df["subject"].astype(str)

    replacements = {
        "Software Construction and Development": "SCD",
        "Lab: Software Construction and Development": "SCD Lab",
        "Formal Methods in Software Engineering": "Formal Methods",
        "Artificial Intelligence": "AI",
        "Information Security": "InfoSec",
        "Professional Practices": "Pro Practices",
        "Web Engineering": "Web Eng",
        "Teachings of Holy Quran": "Quran"
    }

    for full, short in replacements.items():
        df["subject_short"] = df["subject_short"].str.replace(full, short, regex=False)

    return df


def metric_card(title, value, foot, tone="safe"):
    pill_class = "pill-safe" if tone == "safe" else "pill-risk"
    st.markdown(
        f"""
        <div class="metric-wrap">
            <div class="metric-top">{title}</div>
            <div class="metric-value">{value}</div>
            <div class="metric-foot">
                <span class="badge {pill_class}">{foot}</span>
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


def run_step(title, command):
    st.markdown(f"#### {title}")
    log_placeholder = st.empty()
    logs = ""

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=ROOT
    )

    for line in iter(process.stdout.readline, ""):
        logs += line
        log_placeholder.code(logs, language="bash")

    process.stdout.close()
    process.wait()

    if process.returncode == 0:
        st.success(f"{title} completed.")
        return True

    st.error(f"{title} failed.")
    return False


def run_full_pipeline():
    st.markdown('<div class="section-title">Live Sync Console</div>', unsafe_allow_html=True)

    if not run_step("Step 1 • Portal Sync", [sys.executable, "scripts/portal_scraper.py"]):
        return

    if not run_step("Step 2 • Parse Attendance", [sys.executable, "scripts/parse_attendance.py"]):
        return

    if not run_step("Step 3 • Parse Marks", [sys.executable, "scripts/parse_marks.py"]):
        return

    if not run_step("Step 4 • Build Dashboard Dataset", [sys.executable, "scripts/merge_dashboard.py"]):
        return

    st.success("GradeScope sync finished successfully.")
    st.balloons()


def render_sidebar():
    st.sidebar.markdown(
        """
        <div class="sidebar-brand">
            <h1>GradeScope</h1>
            <p>Premium academic analytics for ZABDesk</p>
        </div>
        """,
        unsafe_allow_html=True
    )

    if st.sidebar.button("Dashboard Overview", use_container_width=True):
        set_view("dashboard")
        st.rerun()

    if st.sidebar.button("Launch Portal Sync", use_container_width=True):
        set_view("sync")
        st.rerun()

    if st.sidebar.button("Raw Data Lab", use_container_width=True):
        set_view("raw")
        st.rerun()

    merged_exists = FINAL_DATA_PATH.exists()
    attendance_exists = ATTENDANCE_PATH.exists()
    marks_exists = MARKS_PATH.exists()

    st.sidebar.markdown(
        f"""
        <div class="sidebar-card">
            <div class="sidebar-card-title">Risk Rules</div>
            <div class="sidebar-note">
                <b>Attendance safe zone</b>: 80% or above<br><br>
                <b>Passing marks</b>: 55% or above
            </div>
        </div>
        <div class="sidebar-card">
            <div class="sidebar-card-title">Pipeline Status</div>
            <div class="sidebar-note">
                <b>Merged dashboard</b>: {"Ready" if merged_exists else "Not built yet"}<br><br>
                <b>Attendance summary</b>: {"Ready" if attendance_exists else "Missing"}<br><br>
                <b>Marks summary</b>: {"Ready" if marks_exists else "Missing"}
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


def render_hero():
    st.markdown(
        """
        <div class="hero">
            <div class="hero-badge">Premium Local Dashboard</div>
            <div class="hero-title">GradeScope Portal</div>
            <div class="hero-subtitle">
                A cleaner and faster way to transform raw ZABDesk portal data
                into attendance insights, marks intelligence, risk signals, and action-focused recommendations.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


def render_dashboard(df):
    render_hero()

    if df.empty:
        st.markdown(
            '<div class="surface">No processed dashboard data found yet. Open <b>Launch Portal Sync</b> from the sidebar and run the sync.</div>',
            unsafe_allow_html=True
        )
        return

    total_subjects = len(df)
    high_risk = int((df["final_risk_status"] == "High Risk").sum())
    safe = int((df["final_risk_status"] == "Safe").sum())
    avg_attendance = round(df["attendance_percentage"].mean(), 2)
    avg_marks = round(df["current_marks_percentage"].mean(), 2)

    c1, c2, c3, c4, c5 = st.columns(5)

    with c1:
        metric_card("Subjects", str(total_subjects), "Tracked subjects", "safe")
    with c2:
        metric_card("High Risk", str(high_risk), "Need attention", "risk")
    with c3:
        metric_card("Safe", str(safe), "Currently stable", "safe")
    with c4:
        metric_card("Average Attendance", f"{avg_attendance:.2f}%", "80% safe zone", "safe" if avg_attendance >= 80 else "risk")
    with c5:
        metric_card("Average Marks", f"{avg_marks:.2f}%", "55% passing line", "safe" if avg_marks >= 55 else "risk")

    st.markdown('<div class="section-title">Visual Dashboard</div>', unsafe_allow_html=True)

    risk_color_map = {
        "High Risk": "#D96C5F",
        "Safe": "#7AA889"
    }

    col1, col2 = st.columns(2)

    with col1:
        fig = px.bar(
            df,
            x="subject_short",
            y="attendance_percentage",
            color="final_risk_status",
            color_discrete_map=risk_color_map,
            text="attendance_percentage",
            title="Attendance Health"
        )
        fig.add_hline(y=80, line_dash="dash", line_color="#6F8F72", annotation_text="Acceptable Attendance 80%")
        fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
        fig.update_layout(template="simple_white", height=430, xaxis_title="", yaxis_title="Attendance %", legend_title="")
        fig.update_yaxes(range=[0, 110])
        st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})

    with col2:
        fig = px.bar(
            df,
            x="subject_short",
            y="current_marks_percentage",
            color="final_risk_status",
            color_discrete_map=risk_color_map,
            text="current_marks_percentage",
            title="Marks Performance"
        )
        fig.add_hline(y=55, line_dash="dash", line_color="#7AA1B5", annotation_text="Passing Marks 55%")
        fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside")
        fig.update_layout(template="simple_white", height=430, xaxis_title="", yaxis_title="Marks %", legend_title="")
        fig.update_yaxes(range=[0, 110])
        st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})

    col3, col4 = st.columns(2)

    with col3:
        risk_counts = df["final_risk_status"].value_counts().reset_index()
        risk_counts.columns = ["Status", "Count"]

        fig = px.pie(
            risk_counts,
            names="Status",
            values="Count",
            hole=0.58,
            color="Status",
            color_discrete_map=risk_color_map,
            title="Risk Distribution"
        )
        fig.update_layout(template="simple_white", height=430, legend_title="")
        st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})

    with col4:
        fig = px.scatter(
            df,
            x="attendance_percentage",
            y="current_marks_percentage",
            color="final_risk_status",
            color_discrete_map=risk_color_map,
            text="subject_short",
            size="total_absent",
            title="Risk Map: Attendance vs Marks"
        )
        fig.add_vline(x=80, line_dash="dash", line_color="#6F8F72")
        fig.add_hline(y=55, line_dash="dash", line_color="#7AA1B5")
        fig.update_traces(textposition="top center")
        fig.update_layout(template="simple_white", height=430, xaxis_title="Attendance %", yaxis_title="Marks %", legend_title="")
        fig.update_xaxes(range=[0, 105])
        fig.update_yaxes(range=[0, 105])
        st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})

    st.markdown('<div class="section-title">Performance Wave</div>', unsafe_allow_html=True)

    wave_df = df.sort_values("subject_short").copy()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=wave_df["subject_short"],
        y=wave_df["attendance_percentage"],
        mode="lines+markers",
        name="Attendance %",
        line=dict(color="#7AA889", width=3, shape="spline"),
        marker=dict(size=10)
    ))
    fig.add_trace(go.Scatter(
        x=wave_df["subject_short"],
        y=wave_df["current_marks_percentage"],
        mode="lines+markers",
        name="Marks %",
        line=dict(color="#7AA1B5", width=3, shape="spline"),
        marker=dict(size=10)
    ))
    fig.add_hline(y=80, line_dash="dash", line_color="#6F8F72")
    fig.add_hline(y=55, line_dash="dash", line_color="#7AA1B5")
    fig.update_layout(template="simple_white", height=430, title="Attendance and Marks Wave", xaxis_title="", yaxis_title="Percentage")
    fig.update_yaxes(range=[0, 110])
    st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})

    st.markdown('<div class="section-title">Priority Action Plan</div>', unsafe_allow_html=True)

    risky = df[df["final_risk_status"] == "High Risk"].sort_values(
        by=["attendance_percentage", "current_marks_percentage"]
    )

    if risky.empty:
        st.markdown('<div class="surface">Everything looks stable right now. No subject is currently marked as high risk.</div>', unsafe_allow_html=True)
    else:
        for _, row in risky.iterrows():
            st.markdown(
                f"""
                <div class="callout">
                    <div class="callout-title">{row['subject']}</div>
                    <div class="callout-meta">
                        Attendance: <b>{row['attendance_percentage']:.2f}%</b> •
                        Marks: <b>{row['current_marks_percentage']:.0f}%</b> •
                        Total absents: <b>{int(row['total_absent'])}</b>
                    </div>
                    <div class="callout-body">{row['recommendation']}</div>
                </div>
                """,
                unsafe_allow_html=True
            )

    st.markdown('<div class="section-title">Executive Table</div>', unsafe_allow_html=True)

    dashboard_columns = [
        "subject",
        "attendance_percentage",
        "total_present",
        "total_absent",
        "total_late",
        "quiz_marks",
        "assignment_marks",
        "mid_marks",
        "final_marks",
        "current_marks_percentage",
        "grade",
        "reason",
        "final_risk_status",
        "recommendation"
    ]

    available_columns = [col for col in dashboard_columns if col in df.columns]
    st.dataframe(df[available_columns], use_container_width=True, hide_index=True)


def render_sync_page():
    render_hero()

    st.markdown('<div class="section-title">Portal Sync</div>', unsafe_allow_html=True)

    st.markdown(
        """
        <div class="sync-panel">
            <div class="sync-step">
                <strong>1. Launch</strong><br>
                Click the sync button below. ZABDesk opens in a real browser window.
            </div>
            <div class="sync-step">
                <strong>2. Login manually</strong><br>
                As soon as login is detected, GradeScope starts scraping immediately.
            </div>
            <div class="sync-step">
                <strong>3. Auto pipeline</strong><br>
                Attendance is parsed, marks are parsed, and the dashboard dataset is rebuilt automatically.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    if st.button("Start Live Portal Sync", use_container_width=True):
        run_full_pipeline()


def render_raw_data_page():
    render_hero()

    st.markdown('<div class="section-title">Raw Data Lab</div>', unsafe_allow_html=True)

    attendance_df = load_csv(ATTENDANCE_PATH)
    marks_df = load_csv(MARKS_PATH)
    final_df = load_csv(FINAL_DATA_PATH)

    tabs = st.tabs(["Merged Dashboard", "Attendance Summary", "Marks Summary"])

    with tabs[0]:
        if final_df.empty:
            st.info("No merged dashboard dataset found yet.")
        else:
            st.dataframe(final_df, use_container_width=True, hide_index=True)
            st.download_button("Download merged dashboard CSV", final_df.to_csv(index=False).encode("utf-8"), "gradescope_final_dashboard.csv", "text/csv", use_container_width=True)

    with tabs[1]:
        if attendance_df.empty:
            st.info("No attendance summary found yet.")
        else:
            st.dataframe(attendance_df, use_container_width=True, hide_index=True)
            st.download_button("Download attendance summary CSV", attendance_df.to_csv(index=False).encode("utf-8"), "scraped_attendance_summary.csv", "text/csv", use_container_width=True)

    with tabs[2]:
        if marks_df.empty:
            st.info("No marks summary found yet.")
        else:
            st.dataframe(marks_df, use_container_width=True, hide_index=True)
            st.download_button("Download marks summary CSV", marks_df.to_csv(index=False).encode("utf-8"), "scraped_marks_summary.csv", "text/csv", use_container_width=True)


render_sidebar()

data_df = prepare_df(load_csv(FINAL_DATA_PATH))

if st.session_state.view == "dashboard":
    render_dashboard(data_df)
elif st.session_state.view == "sync":
    render_sync_page()
elif st.session_state.view == "raw":
    render_raw_data_page()
'''

Path("app.py").write_text(app_code, encoding="utf-8")

print("app.py updated successfully.")

app.py updated successfully.


In [4]:
from pathlib import Path

Path("scripts").mkdir(exist_ok=True)

portal_scraper_code = r'''
import re
from pathlib import Path
from playwright.sync_api import sync_playwright

PORTAL_URL = "https://springzabdesk.szabist-isb.edu.pk/"
BASE_URL = "https://springzabdesk.szabist-isb.edu.pk"

ROOT = Path(__file__).resolve().parents[1]
CAPTURE_DIR = ROOT / "data" / "raw" / "portal_captures"
CAPTURE_DIR.mkdir(parents=True, exist_ok=True)


def clean_filename(text):
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    return text.strip("_").lower()


def save_page(page, filename_prefix):
    html_path = CAPTURE_DIR / f"{filename_prefix}.html"
    png_path = CAPTURE_DIR / f"{filename_prefix}.png"

    html_path.write_text(page.content(), encoding="utf-8")
    page.screenshot(path=str(png_path), full_page=True)

    print(f"Saved {html_path.name}")
    print(f"Saved {png_path.name}")


def get_link_href(page, text_value):
    locator = page.locator("a", has_text=text_value).first
    href = locator.get_attribute("href")

    if not href:
        raise ValueError(f"Could not find href for: {text_value}")

    if href.startswith("/"):
        href = BASE_URL + href

    return href


def get_course_links(page):
    course_names = []
    links = page.locator("a").all()

    for link in links:
        text = link.inner_text().strip()
        href = link.get_attribute("href")

        if href and "chkSubmit" in href and text and text not in course_names:
            course_names.append(text)

    return course_names


def wait_for_login(page):
    print("============================================================")
    print("GradeScope Portal Sync")
    print("Login manually in the opened ZABDesk browser.")
    print("Sync starts immediately after login is detected.")
    print("============================================================")

    page.wait_for_selector("a:has-text('View Attendance')", timeout=180000)
    page.wait_for_selector("a:has-text('Current Semester Results')", timeout=180000)

    print("Login detected.")
    print("Starting portal capture...")


def open_course_detail(page, course_name):
    course_link = page.locator("a", has_text=course_name).first
    course_link.click()
    page.wait_for_load_state("domcontentloaded")
    page.wait_for_timeout(900)


def capture_course_details(page, main_url, page_type):
    print(f"Opening {page_type} main page...")
    page.goto(main_url, wait_until="domcontentloaded")
    page.wait_for_timeout(1200)

    save_page(page, f"{page_type}_main_page")

    courses = get_course_links(page)
    print(f"Found {len(courses)} courses on {page_type} page.")

    for index, course_name in enumerate(courses, start=1):
        safe_name = clean_filename(course_name)
        print(f"{page_type.upper()} {index}/{len(courses)} -> {course_name}")

        page.goto(main_url, wait_until="domcontentloaded")
        page.wait_for_timeout(700)

        open_course_detail(page, course_name)
        save_page(page, f"{page_type}_{index}_{safe_name}")

    print(f"{page_type.capitalize()} capture complete.")


def run_scraper():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page(viewport={"width": 1440, "height": 940})

        print("Opening ZABDesk portal...")
        page.goto(PORTAL_URL, wait_until="domcontentloaded")

        wait_for_login(page)

        save_page(page, "logged_in_homepage")

        attendance_url = get_link_href(page, "View Attendance")
        results_url = get_link_href(page, "Current Semester Results")

        print("Attendance URL detected.")
        print("Results URL detected.")

        capture_course_details(page, attendance_url, "attendance")
        capture_course_details(page, results_url, "results")

        browser.close()
        print("Portal sync finished successfully.")


if __name__ == "__main__":
    run_scraper()
'''

Path("scripts/portal_scraper.py").write_text(portal_scraper_code, encoding="utf-8")

print("scripts/portal_scraper.py updated successfully.")

scripts/portal_scraper.py updated successfully.


In [6]:
from pathlib import Path
import zipfile
import fnmatch

ROOT = Path.cwd()
ZIP_NAME = "GradeScope_Public_Release.zip"

exclude_patterns = [
    ".env",
    ".venv/*",
    "__pycache__/*",
    ".ipynb_checkpoints/*",

    "data/raw/*",
    "data/summaries/*",
    "data/processed/final_scraped_academic_dashboard.csv",

    "assets/screenshots/*",

    "*.htm",
    "*.html",
    "GradeScope_Public_Release.zip",
]

allow_patterns = [
    "assets/reports/gradescope_report.html",
]

def should_exclude(path):
    rel = path.as_posix()

    for allow in allow_patterns:
        if fnmatch.fnmatch(rel, allow):
            return False

    for pattern in exclude_patterns:
        if fnmatch.fnmatch(rel, pattern):
            return True

    return False

zip_path = ROOT / ZIP_NAME

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in ROOT.rglob("*"):
        if file_path.is_file():
            relative_path = file_path.relative_to(ROOT)

            if should_exclude(relative_path):
                continue

            zipf.write(file_path, relative_path)

print(f"Created clean ZIP: {ZIP_NAME}")

Created clean ZIP: GradeScope_Public_Release.zip


In [7]:
from pathlib import Path

app_code = r'''
from pathlib import Path
from datetime import datetime
import shutil
import subprocess
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
TEXT_DUMPS_DIR = RAW_DIR / "text_dumps"
CAPTURE_DIR = RAW_DIR / "portal_captures"
SUMMARIES_DIR = DATA_DIR / "summaries"
PROCESSED_DIR = DATA_DIR / "processed"

FINAL_DATA_PATH = PROCESSED_DIR / "gradescope_final_dashboard.csv"
ATTENDANCE_PATH = SUMMARIES_DIR / "scraped_attendance_summary.csv"
MARKS_PATH = SUMMARIES_DIR / "scraped_marks_summary.csv"
SYNC_LOG_PATH = TEXT_DUMPS_DIR / "sync_log.txt"

for folder in [TEXT_DUMPS_DIR, CAPTURE_DIR, SUMMARIES_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

st.set_page_config(
    page_title="GradeScope",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

CUSTOM_CSS = """
<style>
:root {
    --bg: #F8FBF8;
    --surface: #FFFFFF;
    --surface-2: #F2F7F3;
    --line: #E2ECE5;
    --text: #0F172A;
    --muted: #5F6B7A;
    --sage: #8FAE8E;
    --sage-dark: #6C8C70;
    --sage-soft: #E8F1E9;
    --blue: #A9D6E5;
    --blue-dark: #6F99A9;
    --blue-soft: #EFF8FC;
    --danger: #D96C5F;
    --danger-soft: #FCEDEC;
    --gold: #E1B95B;
    --gold-soft: #FBF6E8;
    --shadow: 0 14px 40px rgba(15, 23, 42, 0.07);
    --radius-xl: 28px;
    --radius-lg: 22px;
    --radius-md: 16px;
}

#MainMenu, footer, header {
    visibility: hidden;
}

[data-testid="stHeader"] {
    display: none !important;
}

[data-testid="stToolbar"] {
    display: none !important;
}

[data-testid="stDecoration"] {
    display: none !important;
}

.stApp {
    background:
        radial-gradient(circle at top left, rgba(169,214,229,0.23) 0%, rgba(169,214,229,0) 28%),
        radial-gradient(circle at top right, rgba(143,174,142,0.17) 0%, rgba(143,174,142,0) 28%),
        var(--bg);
    color: var(--text);
}

.block-container {
    padding-top: 1rem !important;
    padding-bottom: 2rem !important;
    max-width: 1320px;
}

section[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #FFFFFF 0%, #F5F9F6 100%);
    border-right: 1px solid #E3ECE7;
}

section[data-testid="stSidebar"] .block-container {
    padding-top: 1rem !important;
    padding-bottom: 1rem !important;
}

h1, h2, h3, h4, h5, h6 {
    color: var(--text);
    letter-spacing: -0.03em;
}

.sidebar-head {
    animation: fadeUp .45s ease;
    padding: 12px 0 8px 0;
    margin-bottom: 12px;
}

.sidebar-mini-title {
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 0.11em;
    text-transform: uppercase;
    color: var(--muted);
    margin-bottom: 4px;
}

.sidebar-mini-copy {
    font-size: 18px;
    font-weight: 900;
    color: var(--text);
}

.sidebar-fact {
    animation: fadeUp .5s ease;
    border: 1px solid var(--line);
    background: linear-gradient(135deg, #FFFFFF 0%, #F6FAF7 100%);
    border-radius: 18px;
    box-shadow: var(--shadow);
    padding: 14px;
    margin-top: 10px;
}

.sidebar-fact-title {
    font-size: 12px;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    color: var(--muted);
    margin-bottom: 8px;
}

.sidebar-fact-body {
    font-size: 13px;
    color: var(--text);
    line-height: 1.55;
}

.streamlit-expanderHeader {
    font-weight: 800 !important;
    font-size: 15px !important;
    color: var(--text) !important;
}

.stButton > button {
    width: 100%;
    min-height: 50px;
    border-radius: 16px;
    border: 1px solid #DCE8E1;
    background: #FFFFFF;
    color: var(--text);
    font-weight: 800;
    font-size: 14px;
    box-shadow: 0 10px 22px rgba(15, 23, 42, 0.05);
    transition: all .2s ease;
}

.stButton > button:hover {
    transform: translateY(-2px);
    border-color: #A8C3B2;
    box-shadow: 0 18px 34px rgba(15, 23, 42, 0.09);
}

.hero {
    animation: fadeUp .5s ease;
    border-radius: 30px;
    border: 1px solid var(--line);
    background: linear-gradient(135deg, #FFFFFF 0%, #F3F8F5 60%, #EEF8FB 100%);
    box-shadow: var(--shadow);
    padding: 34px;
    margin-bottom: 24px;
}

.hero-chip {
    display: inline-block;
    padding: 8px 14px;
    border-radius: 999px;
    background: #E8F0E9;
    color: #48644E;
    font-size: 11px;
    font-weight: 900;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    margin-bottom: 16px;
}

.hero-title {
    font-size: 54px;
    line-height: 1;
    font-weight: 900;
    color: var(--text);
    margin-bottom: 14px;
    letter-spacing: -0.05em;
}

.hero-copy {
    font-size: 18px;
    color: var(--muted);
    line-height: 1.65;
    max-width: 760px;
}

.section-title {
    animation: fadeUp .5s ease;
    font-size: 30px;
    font-weight: 900;
    margin: 20px 0 8px 0;
    letter-spacing: -0.04em;
}

.section-copy {
    animation: fadeUp .55s ease;
    font-size: 15px;
    color: var(--muted);
    margin-bottom: 18px;
}

.surface {
    animation: fadeUp .55s ease;
    border-radius: 24px;
    border: 1px solid var(--line);
    background: var(--surface);
    box-shadow: var(--shadow);
    padding: 22px;
}

.card {
    animation: fadeUp .55s ease;
    border-radius: 22px;
    border: 1px solid var(--line);
    background: linear-gradient(135deg, #FFFFFF 0%, #F9FCFA 100%);
    box-shadow: var(--shadow);
    padding: 20px;
    transition: all .2s ease;
    height: 100%;
}

.card:hover {
    transform: translateY(-2px);
    box-shadow: 0 20px 38px rgba(15, 23, 42, 0.10);
}

.kpi-card {
    animation: fadeUp .55s ease;
    border-radius: 24px;
    border: 1px solid var(--line);
    background: linear-gradient(135deg, #FFFFFF 0%, #F8FCFA 100%);
    box-shadow: var(--shadow);
    padding: 20px;
    min-height: 138px;
    transition: all .2s ease;
}

.kpi-card:hover {
    transform: translateY(-2px);
    box-shadow: 0 20px 38px rgba(15, 23, 42, 0.10);
}

.kpi-label {
    font-size: 12px;
    font-weight: 900;
    color: var(--muted);
    text-transform: uppercase;
    letter-spacing: 0.1em;
    margin-bottom: 14px;
}

.kpi-value {
    font-size: 42px;
    font-weight: 900;
    color: var(--text);
    line-height: 1;
    letter-spacing: -0.04em;
}

.kpi-foot {
    margin-top: 12px;
}

.pill {
    display: inline-block;
    padding: 8px 12px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 0.03em;
}

.pill-safe {
    color: #426646;
    background: #EAF4EC;
    border: 1px solid #D4E7D8;
}

.pill-risk {
    color: #8E433A;
    background: #FCECE9;
    border: 1px solid #F2D2CE;
}

.pill-info {
    color: #446675;
    background: #EEF8FB;
    border: 1px solid #D5EAF2;
}

.fact-grid-card {
    animation: fadeUp .55s ease;
    border-radius: 20px;
    border: 1px solid var(--line);
    background: var(--surface);
    box-shadow: var(--shadow);
    padding: 18px;
    height: 100%;
}

.fact-top {
    font-size: 12px;
    font-weight: 900;
    color: var(--muted);
    text-transform: uppercase;
    letter-spacing: .09em;
    margin-bottom: 10px;
}

.fact-value {
    font-size: 28px;
    font-weight: 900;
    color: var(--text);
    letter-spacing: -0.03em;
    margin-bottom: 8px;
}

.fact-copy {
    font-size: 14px;
    color: var(--muted);
    line-height: 1.6;
}

.chart-card {
    animation: fadeUp .55s ease;
    border-radius: 24px;
    border: 1px solid var(--line);
    background: #FFFFFF;
    box-shadow: var(--shadow);
    padding: 12px 12px 6px 12px;
    margin-bottom: 18px;
}

.chart-title {
    font-size: 18px;
    font-weight: 900;
    color: var(--text);
    padding: 10px 12px 0 12px;
}

.chart-copy {
    font-size: 13px;
    color: var(--muted);
    padding: 0 12px 8px 12px;
}

.callout {
    animation: fadeUp .55s ease;
    border-radius: 20px;
    border: 1px solid var(--line);
    background: #FFFFFF;
    box-shadow: var(--shadow);
    padding: 18px;
    margin-bottom: 14px;
}

.callout-title {
    font-size: 17px;
    font-weight: 900;
    color: var(--text);
    margin-bottom: 8px;
}

.callout-meta {
    font-size: 13px;
    color: var(--muted);
    margin-bottom: 10px;
}

.callout-body {
    font-size: 14px;
    color: var(--text);
    line-height: 1.65;
}

.timeline-card {
    animation: fadeUp .55s ease;
    border-radius: 22px;
    border: 1px solid var(--line);
    background: #FFFFFF;
    box-shadow: var(--shadow);
    padding: 18px;
    height: 100%;
}

.timeline-step {
    border-left: 4px solid #A9D6E5;
    padding: 8px 0 8px 14px;
    margin-bottom: 12px;
    background: #FBFEFF;
    border-radius: 0 14px 14px 0;
}

.footer-wrap {
    animation: fadeUp .55s ease;
    margin-top: 26px;
    padding: 22px 10px 10px 10px;
    border-top: 1px solid var(--line);
    color: var(--muted);
    font-size: 14px;
}

.small-note {
    font-size: 13px;
    color: var(--muted);
    line-height: 1.6;
}

.danger-zone {
    border: 1px solid #F2D2CE;
    background: #FFF8F7;
}

@keyframes fadeUp {
    from {
        opacity: 0;
        transform: translateY(14px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}
</style>
"""

st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

if "page" not in st.session_state:
    st.session_state.page = "home"


def set_page(page_name):
    st.session_state.page = page_name


def load_csv(path):
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            return pd.DataFrame()
    return pd.DataFrame()


def format_time(path):
    if not path.exists():
        return "No local dataset yet"
    timestamp = datetime.fromtimestamp(path.stat().st_mtime)
    return timestamp.strftime("%d %b %Y • %I:%M %p")


def short_subject(text):
    replacements = {
        "Software Construction and Development": "SCD",
        "Lab: Software Construction and Development": "SCD Lab",
        "Formal Methods in Software Engineering": "Formal Methods",
        "Artificial Intelligence": "AI",
        "Information Security": "InfoSec",
        "Professional Practices": "Pro Practices",
        "Web Engineering": "Web Eng",
        "Teachings of Holy Quran": "Quran",
    }
    result = str(text)
    for full, short in replacements.items():
        result = result.replace(full, short)
    return result


def prepare_df(df):
    if df.empty:
        return df

    df = df.copy()

    numeric_columns = [
        "attendance_percentage",
        "total_present",
        "total_absent",
        "total_late",
        "quiz_marks",
        "assignment_marks",
        "mid_marks",
        "final_marks",
        "total_obtained_marks",
        "current_marks_percentage",
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    if "subject" in df.columns:
        df["subject_short"] = df["subject"].apply(short_subject)
    else:
        df["subject_short"] = "Subject"

    return df


def clear_local_data():
    target_paths = [
        FINAL_DATA_PATH,
        ATTENDANCE_PATH,
        MARKS_PATH,
    ]

    for path in target_paths:
        if path.exists():
            path.unlink()

    for folder in [TEXT_DUMPS_DIR, CAPTURE_DIR]:
        if folder.exists():
            for item in folder.iterdir():
                if item.is_file():
                    item.unlink()
                elif item.is_dir():
                    shutil.rmtree(item)

    try:
        st.cache_data.clear()
    except Exception:
        pass

    try:
        st.cache_resource.clear()
    except Exception:
        pass

    keep_keys = {"page"}
    for key in list(st.session_state.keys()):
        if key not in keep_keys:
            del st.session_state[key]


def metric_card(title, value, foot, tone="safe"):
    pill_class = "pill-safe" if tone == "safe" else "pill-risk" if tone == "risk" else "pill-info"
    st.markdown(
        f"""
        <div class="kpi-card">
            <div class="kpi-label">{title}</div>
            <div class="kpi-value">{value}</div>
            <div class="kpi-foot"><span class="pill {pill_class}">{foot}</span></div>
        </div>
        """,
        unsafe_allow_html=True
    )


def fact_card(title, value, copy):
    st.markdown(
        f"""
        <div class="fact-grid-card">
            <div class="fact-top">{title}</div>
            <div class="fact-value">{value}</div>
            <div class="fact-copy">{copy}</div>
        </div>
        """,
        unsafe_allow_html=True
    )


def data_snapshot(df):
    if df.empty:
        return {
            "subjects": 0,
            "high_risk": 0,
            "safe": 0,
            "avg_attendance": 0,
            "avg_marks": 0,
            "below_attendance": 0,
            "below_marks": 0,
            "missing_finals": 0,
            "absents": 0,
        }

    return {
        "subjects": len(df),
        "high_risk": int((df["final_risk_status"] == "High Risk").sum()),
        "safe": int((df["final_risk_status"] == "Safe").sum()),
        "avg_attendance": round(df["attendance_percentage"].mean(), 2),
        "avg_marks": round(df["current_marks_percentage"].mean(), 2),
        "below_attendance": int((df["attendance_percentage"] < 80).sum()),
        "below_marks": int((df["current_marks_percentage"] < 55).sum()),
        "missing_finals": int((df["final_marks"] == 0).sum()) if "final_marks" in df.columns else 0,
        "absents": int(df["total_absent"].sum()) if "total_absent" in df.columns else 0,
    }


def raw_text_files():
    if not TEXT_DUMPS_DIR.exists():
        return []
    return sorted(TEXT_DUMPS_DIR.glob("*.txt"))


def render_sidebar(df):
    snap = data_snapshot(df)

    st.sidebar.markdown(
        """
        <div class="sidebar-head">
            <div class="sidebar-mini-title">Workspace</div>
            <div class="sidebar-mini-copy">GradeScope</div>
        </div>
        """,
        unsafe_allow_html=True
    )

    with st.sidebar.expander("Dashboard", expanded=True):
        if st.button("Landing Page", use_container_width=True, key="nav_home"):
            set_page("home")
            st.rerun()

        if st.button("Insights Dashboard", use_container_width=True, key="nav_dashboard"):
            set_page("dashboard")
            st.rerun()

        if st.button("Portal Sync", use_container_width=True, key="nav_sync"):
            set_page("sync")
            st.rerun()

        if st.button("Raw Sync Notes", use_container_width=True, key="nav_raw"):
            set_page("raw")
            st.rerun()

    with st.sidebar.expander("Settings", expanded=False):
        if st.button("Power Tools", use_container_width=True, key="nav_settings"):
            set_page("settings")
            st.rerun()

    st.sidebar.markdown(
        f"""
        <div class="sidebar-fact">
            <div class="sidebar-fact-title">Risk Rules</div>
            <div class="sidebar-fact-body">
                <b>Attendance safe zone</b>: 80% or above<br><br>
                <b>Passing marks</b>: 55% or above
            </div>
        </div>
        <div class="sidebar-fact">
            <div class="sidebar-fact-title">Live Snapshot</div>
            <div class="sidebar-fact-body">
                <b>Tracked subjects</b>: {snap['subjects']}<br><br>
                <b>High risk subjects</b>: {snap['high_risk']}<br><br>
                <b>Final marks missing</b>: {snap['missing_finals']}
            </div>
        </div>
        <div class="sidebar-fact">
            <div class="sidebar-fact-title">Pipeline Status</div>
            <div class="sidebar-fact-body">
                <b>Merged dashboard</b>: {"Ready" if FINAL_DATA_PATH.exists() else "Missing"}<br><br>
                <b>Attendance summary</b>: {"Ready" if ATTENDANCE_PATH.exists() else "Missing"}<br><br>
                <b>Marks summary</b>: {"Ready" if MARKS_PATH.exists() else "Missing"}
            </div>
        </div>
        <div class="sidebar-fact">
            <div class="sidebar-fact-title">Data Freshness</div>
            <div class="sidebar-fact-body">
                <b>Last dashboard build</b><br>{format_time(FINAL_DATA_PATH)}
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


def render_footer():
    st.markdown(
        """
        <div class="footer-wrap">
            <b>GradeScope</b> • local academic analytics for ZABDesk • built to turn scattered attendance and marks into clear action.
        </div>
        """,
        unsafe_allow_html=True
    )


def render_home(df):
    snap = data_snapshot(df)

    st.markdown(
        """
        <div class="hero">
            <div class="hero-chip">Academic Clarity, Without the Mess</div>
            <div class="hero-title">See trouble early. Fix it before finals do the talking.</div>
            <div class="hero-copy">
                GradeScope pulls your local ZABDesk data into one clean view so you can spot weak marks, short attendance,
                and missing finals without digging through page after page of portal screens.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    c1, c2 = st.columns([1, 1])
    with c1:
        if st.button("Open the dashboard", use_container_width=True, key="home_go_dashboard"):
            set_page("dashboard")
            st.rerun()
    with c2:
        if st.button("Start a fresh portal sync", use_container_width=True, key="home_go_sync"):
            set_page("sync")
            st.rerun()

    st.markdown('<div class="section-title">How it works</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-copy">Three simple steps. No portal chaos, no guesswork.</div>', unsafe_allow_html=True)

    a, b, c = st.columns(3)
    with a:
        fact_card("Step 1", "Connect", "Launch the portal locally and log in yourself. Nothing is stored in the app.")
    with b:
        fact_card("Step 2", "Capture", "GradeScope collects attendance pages, marks pages, and rough raw portal text.")
    with c:
        fact_card("Step 3", "Clarify", "The app turns everything into a clean dashboard with thresholds, risk flags, and next actions.")

    st.markdown('<div class="section-title">What you get</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-copy">Useful outputs only — no filler, no vanity cards.</div>', unsafe_allow_html=True)

    d, e, f = st.columns(3)
    with d:
        fact_card("Attendance", f"{snap['below_attendance']} below 80%", "See which courses are under the safe attendance line.")
    with e:
        fact_card("Marks", f"{snap['below_marks']} below 55%", "Track the courses that are still below the passing threshold.")
    with f:
        fact_card("Finals", f"{snap['missing_finals']} not entered", "Know where your dashboard is incomplete because final marks are still missing.")

    st.markdown('<div class="section-title">Live system snapshot</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-copy">These are live facts pulled from your current local data.</div>', unsafe_allow_html=True)

    g, h, i, j = st.columns(4)
    with g:
        fact_card("Tracked subjects", str(snap["subjects"]), "Subjects currently available in your local dataset.")
    with h:
        fact_card("High risk", str(snap["high_risk"]), "Subjects currently flagged by attendance, marks, or portal reason.")
    with i:
        fact_card("Average attendance", f"{snap['avg_attendance']}%", "Your current average attendance across tracked subjects.")
    with j:
        fact_card("Average marks", f"{snap['avg_marks']}%", "Your current average marks based on the available result pages.")

    render_footer()


def white_layout(fig, height=380):
    fig.update_layout(
        template="simple_white",
        height=height,
        paper_bgcolor="#FFFFFF",
        plot_bgcolor="#FFFFFF",
        font=dict(color="#0F172A"),
        margin=dict(l=20, r=20, t=50, b=20),
        legend_title="",
    )
    fig.update_xaxes(showgrid=False, linecolor="#DDE8E1")
    fig.update_yaxes(gridcolor="#EBF1EC", linecolor="#DDE8E1")
    return fig


def chart_frame(title, copy):
    st.markdown(
        f"""
        <div class="chart-card">
            <div class="chart-title">{title}</div>
            <div class="chart-copy">{copy}</div>
        """,
        unsafe_allow_html=True
    )


def chart_frame_close():
    st.markdown("</div>", unsafe_allow_html=True)


def render_dashboard(df):
    snap = data_snapshot(df)

    st.markdown(
        """
        <div class="hero">
            <div class="hero-chip">Overview</div>
            <div class="hero-title">Your academic picture, cleaned up.</div>
            <div class="hero-copy">
                This dashboard focuses on the things that actually matter: attendance health, marks position,
                missing finals, and the subjects that need action first.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    k1, k2, k3, k4, k5 = st.columns(5)
    with k1:
        metric_card("Subjects", str(snap["subjects"]), "Tracked subjects", "info")
    with k2:
        metric_card("High Risk", str(snap["high_risk"]), "Need attention", "risk")
    with k3:
        metric_card("Safe", str(snap["safe"]), "Currently stable", "safe")
    with k4:
        metric_card("Average Attendance", f"{snap['avg_attendance']}%", "80% safe zone", "safe" if snap["avg_attendance"] >= 80 else "risk")
    with k5:
        metric_card("Average Marks", f"{snap['avg_marks']}%", "55% passing line", "safe" if snap["avg_marks"] >= 55 else "risk")

    st.markdown('<div class="section-title">Live facts</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-copy">A few useful facts worth keeping in sight while you read the charts.</div>', unsafe_allow_html=True)

    f1, f2, f3, f4 = st.columns(4)
    with f1:
        fact_card("Attendance alerts", str(snap["below_attendance"]), "Courses currently below the 80% attendance line.")
    with f2:
        fact_card("Marks alerts", str(snap["below_marks"]), "Courses currently below the 55% passing line.")
    with f3:
        fact_card("Missing finals", str(snap["missing_finals"]), "Result pages where final marks are still not entered.")
    with f4:
        fact_card("Total absents", str(snap["absents"]), "Total absents counted across all tracked attendance pages.")

    if df.empty:
        st.markdown(
            '<div class="surface">There is no local dashboard dataset yet. Go to <b>Portal Sync</b> and run a fresh sync.</div>',
            unsafe_allow_html=True
        )
        render_footer()
        return

    risk_colors = {
        "High Risk": "#D96C5F",
        "Safe": "#82A88A",
    }

    # 1) Attendance Health
    chart_frame(
        "Attendance Health",
        "A subject-level view of attendance. The dashed line marks the 80% safe zone."
    )
    fig = px.bar(
        df,
        x="subject_short",
        y="attendance_percentage",
        color="final_risk_status",
        color_discrete_map=risk_colors,
        text="attendance_percentage"
    )
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
    fig.add_hline(y=80, line_dash="dash", line_color="#6C8C70", annotation_text="80% safe zone")
    fig.update_yaxes(range=[0, 110], title="Attendance %")
    fig.update_xaxes(title="")
    st.plotly_chart(white_layout(fig, height=380), use_container_width=True, config={"displayModeBar": False})
    chart_frame_close()

    # 2) Marks Performance
    chart_frame(
        "Marks Performance",
        "Current marks by subject. The dashed line marks the 55% passing line."
    )
    fig = px.bar(
        df,
        x="subject_short",
        y="current_marks_percentage",
        color="final_risk_status",
        color_discrete_map=risk_colors,
        text="current_marks_percentage"
    )
    fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside")
    fig.add_hline(y=55, line_dash="dash", line_color="#7DA9BC", annotation_text="55% passing line")
    fig.update_yaxes(range=[0, 110], title="Marks %")
    fig.update_xaxes(title="")
    st.plotly_chart(white_layout(fig, height=380), use_container_width=True, config={"displayModeBar": False})
    chart_frame_close()

    # 3) Risk Distribution
    chart_frame(
        "Risk Distribution",
        "A compact breakdown of what is actually driving attention in the current dataset."
    )

    risk_df = pd.DataFrame({
        "Metric": ["High Risk", "Safe", "Below 80% Attendance", "Below 55% Marks"],
        "Count": [
            snap["high_risk"],
            snap["safe"],
            snap["below_attendance"],
            snap["below_marks"],
        ]
    })

    risk_metric_colors = {
        "High Risk": "#D96C5F",
        "Safe": "#82A88A",
        "Below 80% Attendance": "#E1B95B",
        "Below 55% Marks": "#7DA9BC",
    }

    fig = px.bar(
        risk_df,
        x="Count",
        y="Metric",
        orientation="h",
        text="Count",
        color="Metric",
        color_discrete_map=risk_metric_colors
    )
    fig.update_traces(textposition="outside")
    fig.update_xaxes(title="", range=[0, max(1, risk_df["Count"].max() + 1)])
    fig.update_yaxes(title="")
    st.plotly_chart(white_layout(fig, height=330), use_container_width=True, config={"displayModeBar": False})
    chart_frame_close()

    # 4) Risk Map
    chart_frame(
        "Risk Map",
        "Each point is a subject. The vertical line marks 80% attendance and the horizontal line marks 55% marks."
    )

    plot_df = df.copy()
    plot_df["plot_x"] = plot_df["attendance_percentage"]
    plot_df["plot_y"] = plot_df["current_marks_percentage"]

    duplicate_index = plot_df.groupby(["plot_x", "plot_y"]).cumcount()
    x_offsets = [-1.4, -0.7, 0, 0.7, 1.4]
    y_offsets = [1.2, -1.2, 0, 1.8, -1.8]
    text_positions = ["top center", "bottom center", "middle right", "top left", "bottom left"]

    plot_df["plot_x"] = plot_df["plot_x"] + duplicate_index.map(lambda i: x_offsets[i % len(x_offsets)])
    plot_df["plot_y"] = plot_df["plot_y"] + duplicate_index.map(lambda i: y_offsets[i % len(y_offsets)])
    plot_df["text_position"] = duplicate_index.map(lambda i: text_positions[i % len(text_positions)])
    plot_df["marker_size"] = 18 + plot_df["total_absent"].fillna(0) * 3

    fig = go.Figure()

    for status, color in [("Safe", "#82A88A"), ("High Risk", "#D96C5F")]:
        sub = plot_df[plot_df["final_risk_status"] == status]
        if sub.empty:
            continue

        fig.add_trace(go.Scatter(
            x=sub["plot_x"],
            y=sub["plot_y"],
            mode="markers+text",
            text=sub["subject_short"],
            textposition=sub["text_position"],
            name=status,
            marker=dict(
                size=sub["marker_size"],
                color=color,
                line=dict(color="#FFFFFF", width=1.8),
                opacity=0.92
            ),
            customdata=sub[["subject", "attendance_percentage", "current_marks_percentage", "total_absent"]],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Attendance: %{customdata[1]:.1f}%<br>"
                "Marks: %{customdata[2]:.0f}%<br>"
                "Absents: %{customdata[3]:.0f}<extra></extra>"
            )
        ))

    fig.add_vline(x=80, line_dash="dash", line_color="#6C8C70")
    fig.add_hline(y=55, line_dash="dash", line_color="#7DA9BC")

    fig.add_annotation(x=97, y=96, text="strong zone", showarrow=False, font=dict(color="#6C8C70", size=12))
    fig.add_annotation(x=30, y=20, text="watch closely", showarrow=False, font=dict(color="#D96C5F", size=12))

    fig.update_xaxes(range=[0, 102], title="Attendance %")
    fig.update_yaxes(range=[0, 102], title="Marks %")

    st.plotly_chart(white_layout(fig, height=430), use_container_width=True, config={"displayModeBar": False})
    chart_frame_close()

    st.markdown('<div class="section-title">Priority Action Plan</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-copy">These are the subjects that deserve attention first.</div>', unsafe_allow_html=True)

    risky = df[df["final_risk_status"] == "High Risk"].copy()
    risky = risky.sort_values(by=["attendance_percentage", "current_marks_percentage"])

    if risky.empty:
        st.markdown(
            '<div class="surface">Nothing is currently flagged as high risk. Keep maintaining attendance and marks.</div>',
            unsafe_allow_html=True
        )
    else:
        for _, row in risky.iterrows():
            st.markdown(
                f"""
                <div class="callout">
                    <div class="callout-title">{row['subject']}</div>
                    <div class="callout-meta">
                        Attendance: <b>{row['attendance_percentage']:.1f}%</b> •
                        Marks: <b>{row['current_marks_percentage']:.0f}%</b> •
                        Absents: <b>{int(row['total_absent'])}</b>
                    </div>
                    <div class="callout-body">{row['recommendation']}</div>
                </div>
                """,
                unsafe_allow_html=True
            )

    render_footer()


def run_step(title, command):
    st.markdown(f"#### {title}")
    output = st.empty()
    logs = ""

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=ROOT
    )

    for line in iter(process.stdout.readline, ""):
        logs += line
        output.code(logs, language="bash")

    process.stdout.close()
    process.wait()

    if process.returncode == 0:
        st.success(f"{title} completed.")
        return True

    st.error(f"{title} failed.")
    return False


def run_full_pipeline():
    st.markdown('<div class="section-title">Live sync console</div>', unsafe_allow_html=True)

    if not run_step("Step 1 • Portal Sync", [sys.executable, "scripts/portal_scraper.py"]):
        return

    if not run_step("Step 2 • Parse Attendance", [sys.executable, "scripts/parse_attendance.py"]):
        return

    if not run_step("Step 3 • Parse Marks", [sys.executable, "scripts/parse_marks.py"]):
        return

    if not run_step("Step 4 • Build Dashboard Dataset", [sys.executable, "scripts/merge_dashboard.py"]):
        return

    st.success("Portal sync finished successfully.")
    st.balloons()


def render_sync():
    st.markdown(
        """
        <div class="hero">
            <div class="hero-chip">Portal Sync</div>
            <div class="hero-title">Bring in a fresh snapshot.</div>
            <div class="hero-copy">
                Launch ZABDesk, sign in yourself, and let GradeScope do the rest.
                As soon as login is detected, capture begins immediately — no dead waiting screen.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    a, b = st.columns([1.4, 1])
    with a:
        st.markdown(
            """
            <div class="timeline-card">
                <div class="timeline-step"><b>1. Launch the portal</b><br>Open ZABDesk in a real browser window from inside GradeScope.</div>
                <div class="timeline-step"><b>2. Login manually</b><br>As soon as login is detected, the scraper starts collecting pages.</div>
                <div class="timeline-step"><b>3. Parse and merge</b><br>Attendance pages and result pages are converted into one usable dataset.</div>
                <div class="timeline-step"><b>4. Open the dashboard</b><br>Review risks, weak subjects, missing finals, and the raw sync notes.</div>
            </div>
            """,
            unsafe_allow_html=True
        )
    with b:
        st.markdown(
            f"""
            <div class="fact-grid-card">
                <div class="fact-top">Last dashboard build</div>
                <div class="fact-value">{format_time(FINAL_DATA_PATH)}</div>
                <div class="fact-copy">This updates whenever the full pipeline completes successfully.</div>
            </div>
            """,
            unsafe_allow_html=True
        )

    st.markdown("")
    if st.button("Start live portal sync", use_container_width=True):
        run_full_pipeline()

    render_footer()


def render_raw():
    st.markdown(
        """
        <div class="hero">
            <div class="hero-chip">Raw Sync Notes</div>
            <div class="hero-title">See the rough portal text.</div>
            <div class="hero-copy">
                This page shows the raw text snapshots captured during scraping — the unpolished content before the dashboard cleans it up.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    files = raw_text_files()

    if not files:
        st.markdown('<div class="surface">No raw sync text was found yet. Run a portal sync first.</div>', unsafe_allow_html=True)
        render_footer()
        return

    names = [f.name for f in files]
    selected_name = st.selectbox("Choose a raw text file", names, index=0)
    selected_path = TEXT_DUMPS_DIR / selected_name

    raw_text = selected_path.read_text(encoding="utf-8", errors="ignore")

    st.markdown(
        f"""
        <div class="surface">
            <b>Selected file:</b> {selected_name}<br>
            <span class="small-note">This is a rough text dump captured during the portal sync process.</span>
        </div>
        """,
        unsafe_allow_html=True
    )

    st.text_area("Raw captured text", raw_text, height=520)

    st.download_button(
        "Download this raw text file",
        data=raw_text.encode("utf-8"),
        file_name=selected_name,
        mime="text/plain",
        use_container_width=True
    )

    if SYNC_LOG_PATH.exists():
        st.markdown('<div class="section-title">Sync log</div>', unsafe_allow_html=True)
        st.markdown('<div class="section-copy">A plain-text log of the latest scraping session.</div>', unsafe_allow_html=True)
        sync_log = SYNC_LOG_PATH.read_text(encoding="utf-8", errors="ignore")
        st.text_area("Latest sync log", sync_log, height=260)

    render_footer()


def render_settings():
    st.markdown(
        """
        <div class="hero">
            <div class="hero-chip">Settings</div>
            <div class="hero-title">Power tools.</div>
            <div class="hero-copy">
                Use this page for local maintenance. The clear action removes the currently loaded local data and raw text snapshots.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    c1, c2 = st.columns(2)

    with c1:
        st.markdown(
            f"""
            <div class="surface">
                <b>What clear removes</b><br><br>
                • merged dashboard CSV<br>
                • attendance summary CSV<br>
                • marks summary CSV<br>
                • raw text dumps<br>
                • raw portal captures<br><br>
                <span class="small-note">After clearing, the app shows no loaded local dataset until a new sync is run.</span>
            </div>
            """,
            unsafe_allow_html=True
        )

    with c2:
        st.markdown(
            f"""
            <div class="surface danger-zone">
                <b>Current local state</b><br><br>
                Merged dashboard: {"Ready" if FINAL_DATA_PATH.exists() else "Missing"}<br>
                Attendance summary: {"Ready" if ATTENDANCE_PATH.exists() else "Missing"}<br>
                Marks summary: {"Ready" if MARKS_PATH.exists() else "Missing"}<br>
                Raw text files: {len(raw_text_files())}<br><br>
                <span class="small-note">This action only clears local project data, not your scripts or notebook files.</span>
            </div>
            """,
            unsafe_allow_html=True
        )

    st.markdown("")
    confirm = st.checkbox("I understand that clearing will permanently remove the current local loaded data.")
    if st.button("Clear all local data", use_container_width=True):
        if not confirm:
            st.warning("Please confirm first.")
        else:
            clear_local_data()
            st.success("Local data cleared successfully.")
            set_page("home")
            st.rerun()

    render_footer()


df = prepare_df(load_csv(FINAL_DATA_PATH))
render_sidebar(df)

if st.session_state.page == "home":
    render_home(df)
elif st.session_state.page == "dashboard":
    render_dashboard(df)
elif st.session_state.page == "sync":
    render_sync()
elif st.session_state.page == "raw":
    render_raw()
elif st.session_state.page == "settings":
    render_settings()
'''

Path("app.py").write_text(app_code, encoding="utf-8")
print("app.py updated successfully.")

app.py updated successfully.


In [9]:
from pathlib import Path

scraper_code = r'''
import re
from pathlib import Path
from playwright.sync_api import sync_playwright, Error as PlaywrightError

PORTAL_URL = "https://springzabdesk.szabist-isb.edu.pk/"
BASE_URL = "https://springzabdesk.szabist-isb.edu.pk"

ROOT = Path(__file__).resolve().parents[1]
RAW_DIR = ROOT / "data" / "raw"
CAPTURE_DIR = RAW_DIR / "portal_captures"
TEXT_DUMPS_DIR = RAW_DIR / "text_dumps"
SYNC_LOG_PATH = TEXT_DUMPS_DIR / "sync_log.txt"

CAPTURE_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DUMPS_DIR.mkdir(parents=True, exist_ok=True)


def reset_previous_run():
    for folder in [CAPTURE_DIR, TEXT_DUMPS_DIR]:
        for item in folder.iterdir():
            if item.is_file():
                item.unlink()


def log(message):
    print(message)
    with open(SYNC_LOG_PATH, "a", encoding="utf-8") as file:
        file.write(message + "\n")


def clean_filename(text):
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    return text.strip("_").lower()


def extract_visible_text(page):
    try:
        return page.locator("body").inner_text(timeout=5000)
    except Exception:
        return ""


def save_page_artifacts(page, filename_prefix):
    html_path = CAPTURE_DIR / f"{filename_prefix}.html"
    txt_path = TEXT_DUMPS_DIR / f"{filename_prefix}.txt"

    html_path.write_text(page.content(), encoding="utf-8")
    raw_text = extract_visible_text(page)
    txt_path.write_text(raw_text, encoding="utf-8")

    log(f"Saved {html_path.name}")
    log(f"Saved {txt_path.name}")


def get_link_href(page, text_value):
    locator = page.locator("a", has_text=text_value).first
    href = locator.get_attribute("href")

    if not href:
        raise ValueError(f"Could not find href for: {text_value}")

    if href.startswith("/"):
        href = BASE_URL + href

    return href


def get_course_links(page):
    course_names = []
    links = page.locator("a").all()

    for link in links:
        text = link.inner_text().strip()
        href = link.get_attribute("href")

        if href and "chkSubmit" in href and text and text not in course_names:
            course_names.append(text)

    return course_names


def wait_for_login(page):
    log("============================================================")
    log("GradeScope Portal Sync")
    log("Login manually in the opened ZABDesk browser.")
    log("Sync starts immediately after login is detected.")
    log("Do not close the browser while sync is running.")
    log("============================================================")

    try:
        page.wait_for_function(
            """
            () => {
                const links = Array.from(document.querySelectorAll("a"));
                const hasAttendance = links.some(a => a.innerText.includes("View Attendance"));
                const hasResults = links.some(a => a.innerText.includes("Current Semester Results"));
                return hasAttendance && hasResults;
            }
            """,
            timeout=180000
        )
        log("Login detected.")
        log("Starting portal capture...")
    except PlaywrightError:
        raise RuntimeError(
            "Login was not detected. Make sure you logged in successfully and did not close the browser."
        )


def open_course_detail(page, course_name):
    course_link = page.locator("a", has_text=course_name).first
    course_link.click()
    page.wait_for_load_state("domcontentloaded")
    page.wait_for_timeout(700)


def capture_course_details(page, main_url, page_type):
    log(f"Opening {page_type} main page...")
    page.goto(main_url, wait_until="domcontentloaded")
    page.wait_for_timeout(900)

    save_page_artifacts(page, f"{page_type}_main_page")

    courses = get_course_links(page)
    log(f"Found {len(courses)} courses on {page_type} page.")

    for index, course_name in enumerate(courses, start=1):
        safe_name = clean_filename(course_name)
        log(f"{page_type.upper()} {index}/{len(courses)} -> {course_name}")

        page.goto(main_url, wait_until="domcontentloaded")
        page.wait_for_timeout(500)

        open_course_detail(page, course_name)
        save_page_artifacts(page, f"{page_type}_{index}_{safe_name}")

    log(f"{page_type.capitalize()} capture complete.")


def run_scraper():
    reset_previous_run()

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page(viewport={"width": 1440, "height": 940})

        log("Opening ZABDesk portal...")
        page.goto(PORTAL_URL, wait_until="domcontentloaded")

        wait_for_login(page)

        save_page_artifacts(page, "logged_in_homepage")

        attendance_url = get_link_href(page, "View Attendance")
        results_url = get_link_href(page, "Current Semester Results")

        log("Attendance URL detected.")
        log("Results URL detected.")

        capture_course_details(page, attendance_url, "attendance")
        capture_course_details(page, results_url, "results")

        browser.close()
        log("Portal sync finished successfully.")


if __name__ == "__main__":
    run_scraper()
'''

Path("scripts/portal_scraper.py").write_text(scraper_code, encoding="utf-8")
print("scripts/portal_scraper.py updated successfully.")

scripts/portal_scraper.py updated successfully.


In [10]:
from pathlib import Path
import zipfile
import fnmatch

ROOT = Path.cwd()
ZIP_NAME = "GradeScope_Public_Release.zip"

exclude_patterns = [
    ".env",
    ".venv/*",
    "__pycache__/*",
    ".ipynb_checkpoints/*",

    "data/raw/*",
    "data/summaries/*",
    "data/processed/final_scraped_academic_dashboard.csv",

    "assets/screenshots/*",

    "*.htm",
    "*.html",
    "GradeScope_Public_Release.zip",
]

allow_patterns = [
    "assets/reports/gradescope_report.html",
]

def should_exclude(path):
    rel = path.as_posix()

    for allow in allow_patterns:
        if fnmatch.fnmatch(rel, allow):
            return False

    for pattern in exclude_patterns:
        if fnmatch.fnmatch(rel, pattern):
            return True

    return False

zip_path = ROOT / ZIP_NAME

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in ROOT.rglob("*"):
        if file_path.is_file():
            relative_path = file_path.relative_to(ROOT)

            if should_exclude(relative_path):
                continue

            zipf.write(file_path, relative_path)

print(f"Created clean ZIP: {ZIP_NAME}")
print("Right-click the ZIP in JupyterLab and select Download.")

Created clean ZIP: GradeScope_Public_Release.zip
Right-click the ZIP in JupyterLab and select Download.


In [13]:
from pathlib import Path
import zipfile
import fnmatch

ROOT = Path.cwd()
ZIP_NAME = "GradeScope_Public_Release.zip"

# -------------------------------------------------
# 1. Final README.md
# -------------------------------------------------

readme_lines = [
    "# GradeScope",
    "",
    "GradeScope is a local academic dashboard for students who want a clearer view of their attendance, marks, and subject risk without digging through messy portal pages.",
    "",
    "It opens ZABDesk in a real browser, lets the student log in manually, captures attendance and result pages, parses the data, and shows everything inside a clean Streamlit dashboard.",
    "",
    "The project is designed to run locally. No passwords are stored. No portal credentials are required inside the code.",
    "",
    "## Why I built this",
    "",
    "Student portals usually show the data, but they do not explain what needs attention.",
    "",
    "GradeScope focuses on the practical questions:",
    "",
    "- Which subjects are below safe attendance?",
    "- Which subjects are below passing marks?",
    "- Which results are incomplete because final marks are not entered yet?",
    "- Which subjects should be fixed first?",
    "",
    "## Features",
    "",
    "- Local Streamlit dashboard",
    "- Manual ZABDesk login through Playwright",
    "- Attendance page capture",
    "- Result page capture",
    "- HTML parsing with BeautifulSoup",
    "- Raw scraped text view",
    "- Attendance, absents, late count, quiz marks, assignment marks, mid marks, final marks, total marks, grade, and reason extraction",
    "- Risk rules based on attendance and marks",
    "- Live stat cards",
    "- White-background charts",
    "- Subject-level recommendations",
    "- Clear local data tool",
    "- Windows setup and launch scripts",
    "",
    "## Risk rules",
    "",
    "| Metric | Safe level | Risk condition |",
    "|---|---:|---|",
    "| Attendance | 80% or above | Below 80% |",
    "| Marks | 55% or above | Below 55% |",
    "",
    "## Screenshots",
    "",
    "### Executive Summary",
    "",
    "![KPI Summary](assets/charts/kpi_summary.png)",
    "",
    "### Attendance Health",
    "",
    "![Attendance Health](assets/charts/attendance_health_pro.png)",
    "",
    "### Marks Performance",
    "",
    "![Marks Performance](assets/charts/marks_performance_pro.png)",
    "",
    "### Risk Distribution",
    "",
    "![Risk Distribution](assets/charts/risk_distribution_donut.png)",
    "",
    "### Risk Map",
    "",
    "![Risk Map](assets/charts/risk_map_pro.png)",
    "",
    "## How it works",
    "",
    "1. Start the local app.",
    "2. Open the portal sync page.",
    "3. Log in to ZABDesk manually.",
    "4. GradeScope detects the logged-in session.",
    "5. It captures attendance and marks pages.",
    "6. Parsers extract structured data from the captured HTML.",
    "7. The merged dataset is shown in the dashboard.",
    "",
    "## Quick start on Windows",
    "",
    "### 1. Install dependencies",
    "",
    "Double-click:",
    "",
    "```text",
    "setup.bat",
    "```",
    "",
    "This creates a virtual environment, installs the required Python packages, and installs the Playwright browser.",
    "",
    "### 2. Start GradeScope",
    "",
    "Double-click:",
    "",
    "```text",
    "start_gradescope.bat",
    "```",
    "",
    "This opens the local Streamlit app in the browser.",
    "",
    "### 3. Sync portal data",
    "",
    "Inside the app:",
    "",
    "1. Open `Dashboard` from the sidebar.",
    "2. Select `Portal Sync`.",
    "3. Click `Start live portal sync`.",
    "4. Log in when ZABDesk opens.",
    "5. Wait for capture, parsing, and merge to finish.",
    "6. Return to the dashboard.",
    "",
    "## Manual setup",
    "",
    "```bash",
    "pip install -r requirements.txt",
    "python -m playwright install",
    "streamlit run app.py",
    "```",
    "",
    "## Project structure",
    "",
    "```text",
    "gradescope/",
    "├── app.py",
    "├── setup.bat",
    "├── start_gradescope.bat",
    "├── requirements.txt",
    "├── README.md",
    "├── SZABIST_Academic_Dashboard.ipynb",
    "├── scraper.ipynb",
    "├── scripts/",
    "│   ├── portal_scraper.py",
    "│   ├── parse_attendance.py",
    "│   ├── parse_marks.py",
    "│   └── merge_dashboard.py",
    "├── data/",
    "│   └── processed/",
    "│       └── demo_gradescope_dashboard.csv",
    "├── assets/",
    "│   ├── charts/",
    "│   └── reports/",
    "│       └── gradescope_report.html",
    "└── docs/",
    "    ├── DATA_DICTIONARY.md",
    "    └── PRIVACY_CHECKLIST.md",
    "```",
    "",
    "## Important files",
    "",
    "| File | Purpose |",
    "|---|---|",
    "| `app.py` | Main Streamlit app |",
    "| `setup.bat` | Windows setup script |",
    "| `start_gradescope.bat` | Local app launcher |",
    "| `scripts/portal_scraper.py` | Opens ZABDesk and captures portal pages |",
    "| `scripts/parse_attendance.py` | Extracts attendance records |",
    "| `scripts/parse_marks.py` | Extracts marks records |",
    "| `scripts/merge_dashboard.py` | Creates the final dashboard dataset |",
    "| `data/processed/demo_gradescope_dashboard.csv` | Public-safe demo dataset |",
    "| `docs/DATA_DICTIONARY.md` | Dataset column reference |",
    "| `docs/PRIVACY_CHECKLIST.md` | Safety checklist before publishing |",
    "",
    "## Tech stack",
    "",
    "- Python",
    "- Streamlit",
    "- Playwright",
    "- BeautifulSoup",
    "- Pandas",
    "- NumPy",
    "- Plotly",
    "- Matplotlib",
    "- Jupyter Notebook",
    "",
    "## Privacy notes",
    "",
    "GradeScope is meant to run locally.",
    "",
    "Do not publish real portal data. Do not commit login credentials, screenshots, raw HTML captures, or private academic CSV files.",
    "",
    "The public repo should only contain demo or sanitized data.",
    "",
    "Private files and folders are ignored through `.gitignore`, including:",
    "",
    "```text",
    ".env",
    ".venv/",
    "data/raw/",
    "data/summaries/",
    "assets/screenshots/",
    "real scraped dashboard CSV files",
    "portal HTML files",
    "portal text dumps",
    "```",
    "",
    "## Current status",
    "",
    "GradeScope is a working local academic dashboard. It can open the portal, capture pages, parse attendance and marks, merge the results, show a dashboard, display raw sync notes, and clear local data when needed.",
    "",
    "## Possible improvements",
    "",
    "- PDF report export",
    "- Better mobile layout",
    "- Streamlit Cloud demo mode using fake data",
    "- Grade prediction",
    "- Weekly reminder system",
    "- Course priority planner",
    "",
    "## Disclaimer",
    "",
    "This project is for personal academic tracking and portfolio demonstration. Use it responsibly and keep private student data local.",
]

(ROOT / "README.md").write_text("\n".join(readme_lines), encoding="utf-8")
print("README.md updated.")

# -------------------------------------------------
# 2. Final .gitignore
# -------------------------------------------------

gitignore_lines = [
    ".env",
    ".venv/",
    "__pycache__/",
    ".ipynb_checkpoints/",
    "",
    "data/raw/",
    "data/summaries/",
    "data/processed/final_scraped_academic_dashboard.csv",
    "",
    "assets/screenshots/",
    "",
    "*.htm",
    "*.html",
    "!assets/reports/gradescope_report.html",
    "",
    "GradeScope_Public_Release.zip",
]

(ROOT / ".gitignore").write_text("\n".join(gitignore_lines), encoding="utf-8")
print(".gitignore updated.")

# -------------------------------------------------
# 3. Clean ZIP
# -------------------------------------------------

exclude_patterns = [
    ".env",
    ".venv/*",
    "__pycache__/*",
    ".ipynb_checkpoints/*",

    "data/raw/*",
    "data/summaries/*",
    "data/processed/final_scraped_academic_dashboard.csv",

    "assets/screenshots/*",

    "*.htm",
    "*.html",
    "GradeScope_Public_Release.zip",
]

allow_patterns = [
    "assets/reports/gradescope_report.html",
]

def should_exclude(path):
    rel = path.as_posix()

    for allow in allow_patterns:
        if fnmatch.fnmatch(rel, allow):
            return False

    for pattern in exclude_patterns:
        if fnmatch.fnmatch(rel, pattern):
            return True

    return False

zip_path = ROOT / ZIP_NAME

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in ROOT.rglob("*"):
        if file_path.is_file():
            relative_path = file_path.relative_to(ROOT)

            if should_exclude(relative_path):
                continue

            zipf.write(file_path, relative_path)

print(f"Created clean ZIP: {ZIP_NAME}")
print("Right-click the ZIP in JupyterLab and select Download.")

README.md updated.
.gitignore updated.
Created clean ZIP: GradeScope_Public_Release.zip
Right-click the ZIP in JupyterLab and select Download.


In [12]:
from pathlib import Path

app_code = r'''
from pathlib import Path
from datetime import datetime
import shutil
import subprocess
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
TEXT_DUMPS_DIR = RAW_DIR / "text_dumps"
CAPTURE_DIR = RAW_DIR / "portal_captures"
SUMMARIES_DIR = DATA_DIR / "summaries"
PROCESSED_DIR = DATA_DIR / "processed"

FINAL_DATA_PATH = PROCESSED_DIR / "gradescope_final_dashboard.csv"
ATTENDANCE_PATH = SUMMARIES_DIR / "scraped_attendance_summary.csv"
MARKS_PATH = SUMMARIES_DIR / "scraped_marks_summary.csv"
SYNC_LOG_PATH = TEXT_DUMPS_DIR / "sync_log.txt"

for folder in [TEXT_DUMPS_DIR, CAPTURE_DIR, SUMMARIES_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

st.set_page_config(
    page_title="GradeScope",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

CSS = """
<style>
:root {
    --bg: #F7FAF7;
    --paper: #FFFFFF;
    --ink: #0F172A;
    --muted: #64748B;
    --line: #DCE8E2;
    --sage: #8FAE8E;
    --sage-dark: #5F8065;
    --sage-soft: #E9F2EA;
    --blue: #A9D6E5;
    --blue-dark: #5F92A5;
    --blue-soft: #EAF7FB;
    --danger: #D96C5F;
    --danger-soft: #FFF0EE;
    --gold: #D9AF52;
    --gold-soft: #FFF7E2;
    --shadow: 0 22px 60px rgba(15, 23, 42, 0.08);
    --shadow-soft: 0 12px 35px rgba(15, 23, 42, 0.06);
    --radius-xl: 34px;
    --radius-lg: 24px;
    --radius-md: 18px;
}

#MainMenu, footer, header {
    visibility: hidden;
}

[data-testid="stHeader"],
[data-testid="stToolbar"],
[data-testid="stDecoration"],
[data-testid="stStatusWidget"] {
    display: none !important;
}

html {
    scroll-behavior: smooth;
}

.stApp {
    background:
        radial-gradient(circle at 12% 8%, rgba(169, 214, 229, 0.32), transparent 24%),
        radial-gradient(circle at 88% 8%, rgba(143, 174, 142, 0.22), transparent 28%),
        linear-gradient(180deg, #F8FBF8 0%, #F3F8F5 100%);
    color: var(--ink);
}

.block-container {
    padding-top: 1.4rem !important;
    padding-bottom: 3rem !important;
    max-width: 1280px;
}

section[data-testid="stSidebar"] {
    background: transparent !important;
    border: none !important;
    padding: 14px 0 14px 14px;
}

section[data-testid="stSidebar"] > div {
    background: rgba(255, 255, 255, 0.78);
    backdrop-filter: blur(22px);
    border: 1px solid rgba(220, 232, 226, 0.95);
    border-radius: 30px;
    box-shadow: var(--shadow);
    margin: 12px 0 12px 8px;
    max-height: calc(100vh - 24px);
    overflow-y: auto;
}

section[data-testid="stSidebar"] .block-container {
    padding: 22px 20px !important;
}

[data-testid="stSidebarCollapseButton"] {
    color: var(--ink) !important;
}

h1, h2, h3, h4, h5, h6 {
    color: var(--ink);
    letter-spacing: -0.04em;
}

button {
    transition: all 0.22s ease !important;
}

.stButton > button {
    width: 100%;
    min-height: 52px;
    border-radius: 18px;
    border: 1px solid var(--line);
    background: rgba(255, 255, 255, 0.96);
    color: var(--ink);
    font-weight: 850;
    font-size: 14px;
    box-shadow: var(--shadow-soft);
}

.stButton > button:hover {
    transform: translateY(-2px) scale(1.01);
    border-color: #B8D0C2;
    box-shadow: 0 20px 45px rgba(15, 23, 42, 0.10);
}

.stButton > button:active {
    transform: translateY(0) scale(0.99);
}

.streamlit-expanderHeader {
    background: #0F172A !important;
    color: #FFFFFF !important;
    border-radius: 18px !important;
    font-weight: 900 !important;
    min-height: 52px !important;
    padding-left: 14px !important;
    box-shadow: var(--shadow-soft);
}

.streamlit-expanderHeader p {
    color: #FFFFFF !important;
    font-weight: 900 !important;
}

[data-testid="stExpander"] {
    border: none !important;
    background: transparent !important;
    box-shadow: none !important;
}

[data-testid="stExpander"] details {
    border: none !important;
}

.sidebar-logo {
    padding: 18px 16px 14px 16px;
    margin-bottom: 12px;
    border-radius: 24px;
    background:
        radial-gradient(circle at 20% 10%, rgba(169,214,229,0.38), transparent 32%),
        linear-gradient(135deg, #101828 0%, #172033 100%);
    color: white;
    box-shadow: 0 18px 48px rgba(15, 23, 42, 0.18);
    animation: fadeUp 0.45s ease both;
}

.sidebar-kicker {
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 0.14em;
    opacity: 0.72;
    font-weight: 900;
    margin-bottom: 8px;
}

.sidebar-title {
    font-size: 30px;
    line-height: 1;
    font-weight: 950;
    letter-spacing: -0.06em;
}

.sidebar-subtitle {
    margin-top: 10px;
    font-size: 13px;
    line-height: 1.5;
    opacity: 0.78;
}

.sidebar-fact {
    padding: 15px;
    border-radius: 22px;
    border: 1px solid var(--line);
    background: rgba(255, 255, 255, 0.76);
    box-shadow: var(--shadow-soft);
    margin-top: 12px;
    animation: fadeUp 0.55s ease both;
}

.sidebar-fact-title {
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 0.12em;
    color: var(--muted);
    font-weight: 950;
    margin-bottom: 10px;
}

.sidebar-fact-body {
    font-size: 13px;
    color: var(--ink);
    line-height: 1.7;
}

.sidebar-dot {
    display: inline-block;
    width: 9px;
    height: 9px;
    border-radius: 999px;
    margin-right: 6px;
    background: var(--sage);
}

.hero-section {
    min-height: calc(100vh - 60px);
    display: flex;
    align-items: center;
    padding: 28px 0;
}

.hero-shell {
    width: 100%;
    border-radius: 42px;
    border: 1px solid rgba(220, 232, 226, 0.92);
    background:
        radial-gradient(circle at 82% 16%, rgba(169,214,229,0.42), transparent 28%),
        radial-gradient(circle at 8% 88%, rgba(143,174,142,0.24), transparent 28%),
        linear-gradient(135deg, rgba(255,255,255,0.96), rgba(246,251,248,0.92));
    box-shadow: var(--shadow);
    padding: clamp(34px, 6vw, 78px);
    position: relative;
    overflow: hidden;
    animation: fadeUp 0.6s ease both;
}

.hero-shell:before {
    content: "";
    position: absolute;
    width: 420px;
    height: 420px;
    border-radius: 999px;
    right: -170px;
    top: -160px;
    background: rgba(169,214,229,0.25);
    filter: blur(2px);
}

.hero-chip {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 9px 14px;
    border-radius: 999px;
    background: var(--sage-soft);
    color: #496B50;
    font-size: 11px;
    font-weight: 950;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    margin-bottom: 22px;
}

.hero-title {
    max-width: 900px;
    font-size: clamp(48px, 7vw, 86px);
    line-height: 0.94;
    font-weight: 950;
    letter-spacing: -0.07em;
    color: var(--ink);
    margin-bottom: 24px;
}

.hero-copy {
    max-width: 720px;
    color: var(--muted);
    font-size: clamp(17px, 1.4vw, 21px);
    line-height: 1.72;
    margin-bottom: 28px;
}

.hero-actions {
    display: flex;
    gap: 14px;
    flex-wrap: wrap;
    margin-top: 10px;
}

.hero-note {
    margin-top: 24px;
    display: flex;
    gap: 14px;
    flex-wrap: wrap;
}

.mini-pill {
    padding: 10px 13px;
    border-radius: 999px;
    border: 1px solid var(--line);
    background: rgba(255,255,255,0.78);
    font-size: 13px;
    font-weight: 800;
    color: var(--ink);
}

.section-screen {
    min-height: 92vh;
    padding: 42px 0;
    display: flex;
    align-items: center;
}

.section-wrap {
    width: 100%;
}

.section-head {
    margin-bottom: 24px;
}

.section-kicker {
    font-size: 12px;
    text-transform: uppercase;
    letter-spacing: 0.13em;
    font-weight: 950;
    color: var(--sage-dark);
    margin-bottom: 10px;
}

.section-title {
    font-size: clamp(34px, 4vw, 56px);
    line-height: 1;
    letter-spacing: -0.06em;
    font-weight: 950;
    color: var(--ink);
    margin-bottom: 14px;
}

.section-copy {
    max-width: 720px;
    color: var(--muted);
    font-size: 17px;
    line-height: 1.7;
}

.grid-card {
    min-height: 260px;
    height: 100%;
    padding: 26px;
    border-radius: 28px;
    border: 1px solid var(--line);
    background: rgba(255, 255, 255, 0.84);
    box-shadow: var(--shadow);
    transition: all 0.22s ease;
    animation: fadeUp 0.55s ease both;
}

.grid-card:hover {
    transform: translateY(-4px);
    box-shadow: 0 28px 70px rgba(15, 23, 42, 0.10);
}

.card-step {
    font-size: 12px;
    font-weight: 950;
    text-transform: uppercase;
    letter-spacing: 0.12em;
    color: var(--sage-dark);
    margin-bottom: 18px;
}

.card-title {
    font-size: 30px;
    font-weight: 950;
    letter-spacing: -0.05em;
    line-height: 1;
    margin-bottom: 16px;
    color: var(--ink);
}

.card-copy {
    font-size: 15px;
    color: var(--muted);
    line-height: 1.75;
}

.kpi-card {
    min-height: 148px;
    padding: 22px;
    border-radius: 28px;
    border: 1px solid var(--line);
    background: rgba(255,255,255,0.9);
    box-shadow: var(--shadow);
    transition: all 0.22s ease;
    animation: fadeUp 0.55s ease both;
}

.kpi-card:hover {
    transform: translateY(-3px);
}

.kpi-label {
    color: var(--muted);
    font-size: 12px;
    font-weight: 950;
    text-transform: uppercase;
    letter-spacing: 0.12em;
    margin-bottom: 14px;
}

.kpi-value {
    color: var(--ink);
    font-size: 42px;
    line-height: 1;
    font-weight: 950;
    letter-spacing: -0.06em;
}

.kpi-foot {
    margin-top: 13px;
}

.pill {
    display: inline-flex;
    padding: 8px 12px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 900;
}

.pill-safe {
    background: var(--sage-soft);
    color: #456A4B;
    border: 1px solid #D1E5D6;
}

.pill-risk {
    background: var(--danger-soft);
    color: #8E433A;
    border: 1px solid #F0CFCB;
}

.pill-info {
    background: var(--blue-soft);
    color: #416D7E;
    border: 1px solid #D5EAF2;
}

.fact-card {
    min-height: 190px;
    height: 100%;
    padding: 22px;
    border-radius: 26px;
    border: 1px solid var(--line);
    background: rgba(255,255,255,0.86);
    box-shadow: var(--shadow-soft);
    transition: all 0.2s ease;
    animation: fadeUp 0.55s ease both;
}

.fact-card:hover {
    transform: translateY(-3px);
}

.fact-label {
    color: var(--muted);
    font-size: 12px;
    font-weight: 950;
    text-transform: uppercase;
    letter-spacing: 0.12em;
    margin-bottom: 14px;
}

.fact-value {
    color: var(--ink);
    font-size: 30px;
    font-weight: 950;
    letter-spacing: -0.05em;
    margin-bottom: 12px;
}

.fact-copy {
    color: var(--muted);
    line-height: 1.7;
    font-size: 14px;
}

.dashboard-shell {
    padding: 14px 0 34px 0;
}

.dash-title {
    font-size: clamp(40px, 5vw, 62px);
    line-height: 0.98;
    font-weight: 950;
    letter-spacing: -0.07em;
    color: var(--ink);
    margin-bottom: 14px;
}

.dash-copy {
    max-width: 760px;
    color: var(--muted);
    font-size: 17px;
    line-height: 1.7;
    margin-bottom: 28px;
}

.chart-card {
    border-radius: 30px;
    border: 1px solid var(--line);
    background: #FFFFFF;
    box-shadow: var(--shadow);
    padding: 18px 18px 6px 18px;
    margin-bottom: 28px;
    animation: fadeUp 0.55s ease both;
}

.chart-title {
    font-size: 22px;
    font-weight: 950;
    letter-spacing: -0.04em;
    margin-bottom: 6px;
    color: var(--ink);
}

.chart-copy {
    color: var(--muted);
    font-size: 14px;
    line-height: 1.6;
    margin-bottom: 10px;
}

.callout {
    padding: 20px;
    border-radius: 24px;
    border: 1px solid var(--line);
    background: #FFFFFF;
    box-shadow: var(--shadow-soft);
    margin-bottom: 14px;
    animation: fadeUp 0.5s ease both;
}

.callout-title {
    color: var(--ink);
    font-size: 18px;
    font-weight: 950;
    letter-spacing: -0.04em;
    margin-bottom: 8px;
}

.callout-meta {
    color: var(--muted);
    font-size: 13px;
    margin-bottom: 10px;
}

.callout-body {
    color: var(--ink);
    font-size: 14px;
    line-height: 1.7;
}

.surface {
    padding: 24px;
    border-radius: 28px;
    border: 1px solid var(--line);
    background: rgba(255,255,255,0.88);
    box-shadow: var(--shadow);
    animation: fadeUp 0.55s ease both;
}

.danger-zone {
    background: #FFF8F7;
    border-color: #F0CFCB;
}

.footer {
    margin-top: 36px;
    padding: 26px 0 8px 0;
    border-top: 1px solid var(--line);
    color: var(--muted);
    font-size: 14px;
}

.small-note {
    color: var(--muted);
    font-size: 13px;
    line-height: 1.6;
}

[data-testid="stDataFrame"] {
    border-radius: 20px;
    overflow: hidden;
    box-shadow: var(--shadow-soft);
}

textarea {
    border-radius: 18px !important;
}

@keyframes fadeUp {
    from { opacity: 0; transform: translateY(18px); }
    to { opacity: 1; transform: translateY(0); }
}

@keyframes floatSoft {
    0% { transform: translateY(0); }
    50% { transform: translateY(-8px); }
    100% { transform: translateY(0); }
}
</style>
"""

st.markdown(CSS, unsafe_allow_html=True)

if "page" not in st.session_state:
    st.session_state.page = "home"


def set_page(page):
    st.session_state.page = page


def load_csv(path):
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            return pd.DataFrame()
    return pd.DataFrame()


def format_time(path):
    if not path.exists():
        return "No local data"
    return datetime.fromtimestamp(path.stat().st_mtime).strftime("%d %b %Y, %I:%M %p")


def short_subject(value):
    text = str(value)
    replacements = {
        "Software Construction and Development": "SCD",
        "Lab: Software Construction and Development": "SCD Lab",
        "Formal Methods in Software Engineering": "Formal Methods",
        "Artificial Intelligence": "AI",
        "Information Security": "InfoSec",
        "Professional Practices": "Pro Practices",
        "Web Engineering": "Web Eng",
        "Teachings of Holy Quran": "Quran",
    }
    for full, short in replacements.items():
        text = text.replace(full, short)
    return text


def prepare_df(df):
    if df.empty:
        return df

    df = df.copy()

    numeric_columns = [
        "attendance_percentage",
        "total_present",
        "total_absent",
        "total_late",
        "quiz_marks",
        "assignment_marks",
        "mid_marks",
        "final_marks",
        "total_obtained_marks",
        "current_marks_percentage",
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    if "subject" in df.columns:
        df["subject_short"] = df["subject"].apply(short_subject)
    else:
        df["subject_short"] = "Subject"

    return df


def snapshot(df):
    if df.empty:
        return {
            "subjects": 0,
            "high_risk": 0,
            "safe": 0,
            "avg_attendance": 0,
            "avg_marks": 0,
            "below_attendance": 0,
            "below_marks": 0,
            "missing_finals": 0,
            "absents": 0,
        }

    return {
        "subjects": len(df),
        "high_risk": int((df["final_risk_status"] == "High Risk").sum()),
        "safe": int((df["final_risk_status"] == "Safe").sum()),
        "avg_attendance": round(df["attendance_percentage"].mean(), 2),
        "avg_marks": round(df["current_marks_percentage"].mean(), 2),
        "below_attendance": int((df["attendance_percentage"] < 80).sum()),
        "below_marks": int((df["current_marks_percentage"] < 55).sum()),
        "missing_finals": int((df["final_marks"] == 0).sum()) if "final_marks" in df.columns else 0,
        "absents": int(df["total_absent"].sum()) if "total_absent" in df.columns else 0,
    }


def raw_text_files():
    if not TEXT_DUMPS_DIR.exists():
        return []
    return sorted(TEXT_DUMPS_DIR.glob("*.txt"))


def clear_local_data():
    for path in [FINAL_DATA_PATH, ATTENDANCE_PATH, MARKS_PATH]:
        if path.exists():
            path.unlink()

    for folder in [TEXT_DUMPS_DIR, CAPTURE_DIR]:
        if folder.exists():
            for item in folder.iterdir():
                if item.is_file():
                    item.unlink()
                elif item.is_dir():
                    shutil.rmtree(item)

    try:
        st.cache_data.clear()
    except Exception:
        pass

    for key in list(st.session_state.keys()):
        if key != "page":
            del st.session_state[key]


def kpi(label, value, foot, tone="info"):
    cls = "pill-info"
    if tone == "safe":
        cls = "pill-safe"
    elif tone == "risk":
        cls = "pill-risk"

    st.markdown(
        f"""
        <div class="kpi-card">
            <div class="kpi-label">{label}</div>
            <div class="kpi-value">{value}</div>
            <div class="kpi-foot"><span class="pill {cls}">{foot}</span></div>
        </div>
        """,
        unsafe_allow_html=True
    )


def fact(label, value, copy):
    st.markdown(
        f"""
        <div class="fact-card">
            <div class="fact-label">{label}</div>
            <div class="fact-value">{value}</div>
            <div class="fact-copy">{copy}</div>
        </div>
        """,
        unsafe_allow_html=True
    )


def render_sidebar(df):
    snap = snapshot(df)

    st.sidebar.markdown(
        """
        <div class="sidebar-logo">
            <div class="sidebar-kicker">Workspace</div>
            <div class="sidebar-title">GradeScope</div>
            <div class="sidebar-subtitle">Local academic intelligence for ZABDesk.</div>
        </div>
        """,
        unsafe_allow_html=True
    )

    with st.sidebar.expander("Dashboard", expanded=True):
        if st.button("Landing Page", key="go_home"):
            set_page("home")
            st.rerun()

        if st.button("Insights Dashboard", key="go_dashboard"):
            set_page("dashboard")
            st.rerun()

        if st.button("Portal Sync", key="go_sync"):
            set_page("sync")
            st.rerun()

        if st.button("Raw Sync Notes", key="go_raw"):
            set_page("raw")
            st.rerun()

    with st.sidebar.expander("Settings", expanded=False):
        if st.button("Power Tools", key="go_settings"):
            set_page("settings")
            st.rerun()

    st.sidebar.markdown(
        f"""
        <div class="sidebar-fact">
            <div class="sidebar-fact-title">Risk Rules</div>
            <div class="sidebar-fact-body">
                <span class="sidebar-dot"></span><b>Attendance:</b> 80% safe zone<br>
                <span class="sidebar-dot"></span><b>Marks:</b> 55% passing line
            </div>
        </div>

        <div class="sidebar-fact">
            <div class="sidebar-fact-title">Live Snapshot</div>
            <div class="sidebar-fact-body">
                <b>Subjects:</b> {snap["subjects"]}<br>
                <b>High risk:</b> {snap["high_risk"]}<br>
                <b>Missing finals:</b> {snap["missing_finals"]}
            </div>
        </div>

        <div class="sidebar-fact">
            <div class="sidebar-fact-title">Pipeline</div>
            <div class="sidebar-fact-body">
                <b>Dashboard:</b> {"Ready" if FINAL_DATA_PATH.exists() else "Missing"}<br>
                <b>Attendance:</b> {"Ready" if ATTENDANCE_PATH.exists() else "Missing"}<br>
                <b>Marks:</b> {"Ready" if MARKS_PATH.exists() else "Missing"}
            </div>
        </div>

        <div class="sidebar-fact">
            <div class="sidebar-fact-title">Freshness</div>
            <div class="sidebar-fact-body">
                {format_time(FINAL_DATA_PATH)}
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )


def footer():
    st.markdown(
        """
        <div class="footer">
            <b>GradeScope</b> - local academic tracking, portal sync, and risk clarity.
        </div>
        """,
        unsafe_allow_html=True
    )


def render_home(df):
    snap = snapshot(df)

    st.markdown(
        """
        <section class="hero-section">
            <div class="hero-shell">
                <div class="hero-chip">Academic clarity, minus the portal mess</div>
                <div class="hero-title">Know what needs fixing before it becomes a problem.</div>
                <div class="hero-copy">
                    GradeScope turns ZABDesk attendance and marks into a clean local dashboard.
                    Sync once, review the risks, and focus on the subjects that actually need work.
                </div>
                <div class="hero-note">
                    <div class="mini-pill">Manual login</div>
                    <div class="mini-pill">Local data only</div>
                    <div class="mini-pill">80% attendance rule</div>
                    <div class="mini-pill">55% passing rule</div>
                </div>
            </div>
        </section>
        """,
        unsafe_allow_html=True
    )

    c1, c2 = st.columns(2)
    with c1:
        if st.button("Open insights dashboard", key="hero_dashboard"):
            set_page("dashboard")
            st.rerun()
    with c2:
        if st.button("Start portal sync", key="hero_sync"):
            set_page("sync")
            st.rerun()

    st.markdown(
        """
        <section class="section-screen">
            <div class="section-wrap">
                <div class="section-head">
                    <div class="section-kicker">Flow</div>
                    <div class="section-title">From portal pages to useful decisions.</div>
                    <div class="section-copy">
                        GradeScope keeps the process simple. It captures the data, cleans the data, and shows what matters.
                    </div>
                </div>
            </div>
        </section>
        """,
        unsafe_allow_html=True
    )

    a, b, c = st.columns(3)
    with a:
        st.markdown(
            """
            <div class="grid-card">
                <div class="card-step">Step 1</div>
                <div class="card-title">Connect</div>
                <div class="card-copy">
                    Open ZABDesk from the app and log in manually. No password is saved.
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )
    with b:
        st.markdown(
            """
            <div class="grid-card">
                <div class="card-step">Step 2</div>
                <div class="card-title">Capture</div>
                <div class="card-copy">
                    Attendance pages, result pages, and raw portal text are collected locally.
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )
    with c:
        st.markdown(
            """
            <div class="grid-card">
                <div class="card-step">Step 3</div>
                <div class="card-title">Act</div>
                <div class="card-copy">
                    The dashboard ranks risks and gives practical subject-level recommendations.
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

    st.markdown(
        """
        <section class="section-screen">
            <div class="section-wrap">
                <div class="section-head">
                    <div class="section-kicker">Current Data</div>
                    <div class="section-title">Live facts from your local dashboard.</div>
                    <div class="section-copy">
                        These values update when you run a new sync. No guesswork, no manually checking every course page.
                    </div>
                </div>
            </div>
        </section>
        """,
        unsafe_allow_html=True
    )

    x1, x2, x3, x4 = st.columns(4)
    with x1:
        fact("Subjects", str(snap["subjects"]), "Courses currently available in the local dataset.")
    with x2:
        fact("Attendance alerts", str(snap["below_attendance"]), "Subjects below the 80% safe attendance line.")
    with x3:
        fact("Marks alerts", str(snap["below_marks"]), "Subjects below the 55% passing line.")
    with x4:
        fact("Missing finals", str(snap["missing_finals"]), "Subjects where final marks are still not entered.")

    footer()


def white_chart(fig, height=430):
    fig.update_layout(
        template="simple_white",
        height=height,
        paper_bgcolor="#FFFFFF",
        plot_bgcolor="#FFFFFF",
        font=dict(color="#0F172A"),
        margin=dict(l=20, r=20, t=55, b=25),
        legend_title="",
    )
    fig.update_xaxes(showgrid=False, linecolor="#DCE8E2")
    fig.update_yaxes(gridcolor="#EDF3EF", linecolor="#DCE8E2")
    return fig


def chart_header(title, copy):
    st.markdown(
        f"""
        <div class="chart-card">
            <div class="chart-title">{title}</div>
            <div class="chart-copy">{copy}</div>
        """,
        unsafe_allow_html=True
    )


def chart_close():
    st.markdown("</div>", unsafe_allow_html=True)


def render_dashboard(df):
    snap = snapshot(df)

    st.markdown(
        """
        <div class="dashboard-shell">
            <div class="dash-title">Insights dashboard</div>
            <div class="dash-copy">
                A focused view of attendance, marks, missing finals, and risk priority across your current subjects.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    k1, k2, k3, k4, k5 = st.columns(5)
    with k1:
        kpi("Subjects", str(snap["subjects"]), "tracked", "info")
    with k2:
        kpi("High Risk", str(snap["high_risk"]), "needs action", "risk")
    with k3:
        kpi("Safe", str(snap["safe"]), "stable", "safe")
    with k4:
        kpi("Attendance", f"{snap['avg_attendance']}%", "80% target", "safe" if snap["avg_attendance"] >= 80 else "risk")
    with k5:
        kpi("Marks", f"{snap['avg_marks']}%", "55% target", "safe" if snap["avg_marks"] >= 55 else "risk")

    if df.empty:
        st.markdown('<div class="surface">No local dataset found. Run Portal Sync first.</div>', unsafe_allow_html=True)
        footer()
        return

    colors = {"High Risk": "#D96C5F", "Safe": "#82A88A"}

    chart_header("Attendance Health", "Subject attendance with the 80% safe line.")
    fig = px.bar(df, x="subject_short", y="attendance_percentage", color="final_risk_status", color_discrete_map=colors, text="attendance_percentage")
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
    fig.add_hline(y=80, line_dash="dash", line_color="#5F8065", annotation_text="80% safe")
    fig.update_yaxes(range=[0, 110], title="Attendance %")
    fig.update_xaxes(title="")
    st.plotly_chart(white_chart(fig), use_container_width=True, config={"displayModeBar": False})
    chart_close()

    chart_header("Marks Performance", "Current marks with the 55% passing line.")
    fig = px.bar(df, x="subject_short", y="current_marks_percentage", color="final_risk_status", color_discrete_map=colors, text="current_marks_percentage")
    fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside")
    fig.add_hline(y=55, line_dash="dash", line_color="#5F92A5", annotation_text="55% passing")
    fig.update_yaxes(range=[0, 110], title="Marks %")
    fig.update_xaxes(title="")
    st.plotly_chart(white_chart(fig), use_container_width=True, config={"displayModeBar": False})
    chart_close()

    chart_header("Risk Breakdown", "Counts that explain why subjects are being flagged.")
    risk_df = pd.DataFrame({
        "Metric": ["High Risk", "Safe", "Below 80% Attendance", "Below 55% Marks"],
        "Count": [snap["high_risk"], snap["safe"], snap["below_attendance"], snap["below_marks"]],
    })
    fig = px.bar(
        risk_df,
        x="Count",
        y="Metric",
        orientation="h",
        text="Count",
        color="Metric",
        color_discrete_map={
            "High Risk": "#D96C5F",
            "Safe": "#82A88A",
            "Below 80% Attendance": "#D9AF52",
            "Below 55% Marks": "#5F92A5",
        },
    )
    fig.update_traces(textposition="outside")
    fig.update_xaxes(range=[0, max(1, risk_df["Count"].max() + 1)], title="")
    fig.update_yaxes(title="")
    st.plotly_chart(white_chart(fig, height=340), use_container_width=True, config={"displayModeBar": False})
    chart_close()

    chart_header("Risk Map", "Each point is a subject. The guide lines mark 80% attendance and 55% marks.")
    plot_df = df.copy()
    duplicates = plot_df.groupby(["attendance_percentage", "current_marks_percentage"]).cumcount()
    offset_x = [-1.6, -0.8, 0, 0.8, 1.6]
    offset_y = [1.5, -1.5, 0, 2.2, -2.2]
    plot_df["x"] = plot_df["attendance_percentage"] + duplicates.map(lambda i: offset_x[i % len(offset_x)])
    plot_df["y"] = plot_df["current_marks_percentage"] + duplicates.map(lambda i: offset_y[i % len(offset_y)])
    plot_df["size"] = 18 + plot_df["total_absent"].fillna(0) * 3

    fig = go.Figure()
    for status, color in [("Safe", "#82A88A"), ("High Risk", "#D96C5F")]:
        sub = plot_df[plot_df["final_risk_status"] == status]
        fig.add_trace(go.Scatter(
            x=sub["x"],
            y=sub["y"],
            mode="markers+text",
            text=sub["subject_short"],
            textposition="top center",
            name=status,
            marker=dict(size=sub["size"], color=color, opacity=0.9, line=dict(color="#FFFFFF", width=2)),
            customdata=sub[["subject", "attendance_percentage", "current_marks_percentage", "total_absent"]],
            hovertemplate="<b>%{customdata[0]}</b><br>Attendance: %{customdata[1]:.1f}%<br>Marks: %{customdata[2]:.0f}%<br>Absents: %{customdata[3]}<extra></extra>",
        ))

    fig.add_vline(x=80, line_dash="dash", line_color="#5F8065")
    fig.add_hline(y=55, line_dash="dash", line_color="#5F92A5")
    fig.update_xaxes(range=[0, 105], title="Attendance %")
    fig.update_yaxes(range=[0, 105], title="Marks %")
    st.plotly_chart(white_chart(fig, height=470), use_container_width=True, config={"displayModeBar": False})
    chart_close()

    st.markdown('<div class="section-title">Priority action plan</div>', unsafe_allow_html=True)
    risky = df[df["final_risk_status"] == "High Risk"].sort_values(["attendance_percentage", "current_marks_percentage"])
    if risky.empty:
        st.markdown('<div class="surface">No high-risk subjects right now.</div>', unsafe_allow_html=True)
    else:
        for _, row in risky.iterrows():
            st.markdown(
                f"""
                <div class="callout">
                    <div class="callout-title">{row['subject']}</div>
                    <div class="callout-meta">Attendance: <b>{row['attendance_percentage']:.1f}%</b> • Marks: <b>{row['current_marks_percentage']:.0f}%</b> • Absents: <b>{int(row['total_absent'])}</b></div>
                    <div class="callout-body">{row['recommendation']}</div>
                </div>
                """,
                unsafe_allow_html=True
            )

    footer()


def run_step(title, command):
    st.markdown(f"#### {title}")
    holder = st.empty()
    logs = ""

    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)

    for line in iter(process.stdout.readline, ""):
        logs += line
        holder.code(logs, language="bash")

    process.stdout.close()
    process.wait()

    if process.returncode == 0:
        st.success(f"{title} completed.")
        return True

    st.error(f"{title} failed.")
    return False


def render_sync():
    st.markdown(
        """
        <section class="hero-section">
            <div class="hero-shell">
                <div class="hero-chip">Portal Sync</div>
                <div class="hero-title">Pull in a fresh academic snapshot.</div>
                <div class="hero-copy">
                    Open ZABDesk, log in manually, and GradeScope starts capturing as soon as the session is detected.
                    No credentials are stored in the app.
                </div>
            </div>
        </section>
        """,
        unsafe_allow_html=True
    )

    if st.button("Start live portal sync", key="start_sync"):
        if not run_step("Step 1 • Portal Sync", [sys.executable, "scripts/portal_scraper.py"]):
            return
        if not run_step("Step 2 • Parse Attendance", [sys.executable, "scripts/parse_attendance.py"]):
            return
        if not run_step("Step 3 • Parse Marks", [sys.executable, "scripts/parse_marks.py"]):
            return
        if not run_step("Step 4 • Build Dataset", [sys.executable, "scripts/merge_dashboard.py"]):
            return

        st.success("Sync completed.")
        st.balloons()

    footer()


def render_raw():
    st.markdown(
        """
        <div class="dashboard-shell">
            <div class="dash-title">Raw sync notes</div>
            <div class="dash-copy">
                Rough text captured from the portal before it is cleaned, parsed, and merged.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    files = raw_text_files()
    if not files:
        st.markdown('<div class="surface">No raw sync text found. Run a portal sync first.</div>', unsafe_allow_html=True)
        footer()
        return

    selected = st.selectbox("Select raw text file", [f.name for f in files])
    path = TEXT_DUMPS_DIR / selected
    text = path.read_text(encoding="utf-8", errors="ignore")

    st.text_area("Raw captured text", text, height=560)
    st.download_button("Download selected text", text.encode("utf-8"), selected, "text/plain")

    if SYNC_LOG_PATH.exists():
        st.markdown('<div class="section-title">Latest sync log</div>', unsafe_allow_html=True)
        st.text_area("Sync log", SYNC_LOG_PATH.read_text(encoding="utf-8", errors="ignore"), height=260)

    footer()


def render_settings():
    st.markdown(
        """
        <div class="dashboard-shell">
            <div class="dash-title">Settings</div>
            <div class="dash-copy">
                Maintenance tools for clearing local data and resetting the dashboard state.
            </div>
        </div>
        """,
        unsafe_allow_html=True
    )

    a, b = st.columns(2)
    with a:
        st.markdown(
            """
            <div class="surface">
                <b>Clear local data removes:</b><br><br>
                • merged dashboard CSV<br>
                • attendance summary CSV<br>
                • marks summary CSV<br>
                • raw portal captures<br>
                • raw text dumps<br><br>
                <span class="small-note">It does not delete scripts, notebooks, setup files, or charts.</span>
            </div>
            """,
            unsafe_allow_html=True
        )

    with b:
        st.markdown(
            f"""
            <div class="surface danger-zone">
                <b>Current local state</b><br><br>
                Merged dashboard: {"Ready" if FINAL_DATA_PATH.exists() else "Missing"}<br>
                Attendance summary: {"Ready" if ATTENDANCE_PATH.exists() else "Missing"}<br>
                Marks summary: {"Ready" if MARKS_PATH.exists() else "Missing"}<br>
                Raw text files: {len(raw_text_files())}<br>
            </div>
            """,
            unsafe_allow_html=True
        )

    confirm = st.checkbox("I understand this will remove the current local loaded data.")
    if st.button("Clear all local data", key="clear_local_data"):
        if not confirm:
            st.warning("Confirm first.")
        else:
            clear_local_data()
            st.success("Local data cleared.")
            set_page("home")
            st.rerun()

    footer()


df = prepare_df(load_csv(FINAL_DATA_PATH))
render_sidebar(df)

if st.session_state.page == "home":
    render_home(df)
elif st.session_state.page == "dashboard":
    render_dashboard(df)
elif st.session_state.page == "sync":
    render_sync()
elif st.session_state.page == "raw":
    render_raw()
elif st.session_state.page == "settings":
    render_settings()
'''

Path("app.py").write_text(app_code, encoding="utf-8")
print("app.py updated with full UI overhaul.")

app.py updated with full UI overhaul.


In [14]:
from pathlib import Path

text = Path("app.py").read_text(encoding="utf-8")

checks = [
    "Know what needs fixing before it becomes a problem",
    "floating sidebar",
    "Clear all local data",
    "Raw sync notes",
]

for item in checks:
    print(item, "FOUND" if item in text else "MISSING")

Know what needs fixing before it becomes a problem FOUND
floating sidebar MISSING
Clear all local data FOUND
Raw sync notes FOUND
